# SDIC replication on the unused NEU-CLS images

This is `PHASE0_11_family_uncertainty_revision.ipynb` with one change: section 1 selects, in each class, the images that the paper's 900-image subset did **not** use (150 per class, 900 in all). Corruption families, backbone, calibration arms, folds, seeds and analyses are unchanged, so the run is a replication on non-overlapping images with the same six families. In the output tables the run is still labelled `G1`; the images are different.

**How to run**: *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. NEU-CLS is downloaded automatically from Kaggle. The run takes about 20 minutes. At the end the browser downloads `sdic_replication_other900.zip`; send that file back. It contains the per-image tables, the report, `replication_subset_files.csv` (the file name of every image used) and `original_subset_files.csv` (the paper's subset, for the disjointness check).


# PHASE 0.11 — Family-level uncertainty

## Phase 0.10 found something bigger than the verdict bug it was built to fix

The nested family holdout did not work either. **No legitimate selection rule ranks arms
consistently with the test outcome:**

| Selection rule | Spearman with test NLL |
|---|---|
| nested — 4 families fit, 2 held out | **+0.314** |
| strict — all 6 train families | **+0.029** |
| optimistic — the 6 test families | +1.000 (but it selects on what it reports) |

The held-out selection families produced NLL of 2.05–2.42 while the test families produced
0.76–1.03 — roughly 2.5× harder. **Corruption families are not exchangeable.** A rule tuned on
some families cannot rank calibrators for others, and no amount of nesting fixes that.

## And the results themselves are not stable

Between Phase 0.9 and Phase 0.10 the only change was fitting the calibrators on four of the six
train families instead of all six. Same data, same seed, same folds. C0 and C1 are byte-identical
across the two runs, confirming the difference is entirely the calibration family mix:

| Arm | NLL 0.9 → 0.10 | ECE 0.9 → 0.10 |
|---|---|---|
| C2 scalar / augmented | 0.7915 → 0.7793 | 0.0971 → **0.0382** (−61%) |
| C4 linear / raw *q* | 0.7352 → 0.7885 | 0.0770 → 0.0331 |
| C7q bounded / raw *q* | 0.7545 → 0.7630 | 0.0809 → 0.0329 |
| C5 linear / class one-hot | 0.8841 → 0.8629 | 0.0492 → 0.0432 |

Rank stability across the two runs: **+0.857 on NLL**, and **−0.595 on ECE** — the ECE ordering
essentially reversed. Which arm looks best on calibration error depends on which corruptions the
calibrator happened to be fitted on.

## What this means, and it is the most important finding in the whole audit

Every confidence interval reported so far resamples **images within a fixed set of six test
families**. Those intervals are ±0.005 on NLL and they are not wrong — they are answering a
narrow question: *would this hold on different images of these six corruptions?*

The paper asks a different question: *would this hold on six different corruptions?* For that,
the effective sample size is **n = 6**, not n = 5,400, and the family-to-family spread is large
enough that the answer may well be no.

**Reporting the image-level interval as if it supported a generalisation claim would have been
the single most serious error in the manuscript**, and it would have survived review — the
numbers look rigorous. Phase 0 caught it before any of it was written up. That is worth more than
any of the verdicts that came before.

## What this notebook does

Metrics are recorded **per test family**, and every comparison now carries two intervals: the
image-level bootstrap (conditional on this family set) and a family-level *t* interval over the
six families with 5 degrees of freedom. Comparisons that are significant on images but not across
families are listed explicitly, with their six per-family deltas printed.

The verdict has a new top branch. If **no legitimate selection rule reaches a rank correlation of
0.60** with the test outcome, it returns `UNDETERMINED` and names no method — because reading a
verdict off a misspecified selection is exactly how Phase 0.6, 0.7, 0.9 and 0.10 each produced a
wrong conclusion. A fifth repetition of that mistake is not worth risking.

## 0. Self-contained core module

In [21]:
CORE_V2 = r"""
import math, hashlib
import numpy as np, cv2

def _clip(x): return np.clip(x, 0, 1)
def _mask3(mask, x): return mask[..., None] if x.ndim == 3 else mask

# ------------------------------------------------------------------ optics --
def defocus_blur(x, s):
    r = [1, 2, 3, 5, 7][s-1]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)).astype(np.float32)
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def motion_blur(x, s):
    ksz = [5, 9, 13, 19, 25][s-1]
    ang = np.random.uniform(0, 180)
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), ang, 1.0)
    k = cv2.warpAffine(k, M, (ksz, ksz))
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def vignetting(x, s):
    st = [0.15, 0.30, 0.45, 0.62, 0.80][s-1]
    h, w = x.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((xx-w/2)/(w/2))**2 + ((yy-h/2)/(h/2))**2)
    m = np.clip(1 - st*np.clip(r-0.4, 0, None)/0.6, 0, 1).astype(np.float32)
    return _clip(x * _mask3(m, x))

def lens_contamination(x, s):
    n = [3, 7, 13, 22, 34][s-1]; h, w = x.shape[:2]
    blurred = cv2.GaussianBlur(x, (0, 0), sigmaX=max(1.0, min(h, w)/40))
    mask = np.zeros((h, w), np.float32)
    for _ in range(n):
        c = (np.random.randint(0, w), np.random.randint(0, h))
        rad = np.random.randint(max(2, int(0.015*w)), max(4, int(0.07*w)))
        cv2.circle(mask, c, rad, 1.0, -1, lineType=cv2.LINE_AA)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(1.0, min(h, w)/60))
    m = _mask3(mask, x)                       # guard: 2-D input used to broadcast to (H,W,W)
    return _clip((x*(1-m) + blurred*m) * (1 - 0.25*m))

def vibration_jitter(x, s):
    amp = [0.6, 1.4, 2.6, 4.2, 6.5][s-1]; h, w = x.shape[:2]
    dx, dy = np.random.uniform(-amp, amp, 2)
    out = cv2.warpAffine(x, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    ksz = int(max(3, 2*round(amp)+1))
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M2 = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), math.degrees(math.atan2(dy, dx)), 1.0)
    k = cv2.warpAffine(k, M2, (ksz, ksz))
    return _clip(cv2.filter2D(out, -1, k/max(k.sum(), 1e-8)))

# ------------------------------------------------------------------ sensor --
def gaussian_noise(x, s):
    return _clip(x + np.random.normal(0, [0.03, 0.06, 0.10, 0.16, 0.24][s-1], x.shape))

def shot_noise(x, s):
    lam = [80, 35, 15, 7, 3][s-1]
    return _clip(np.random.poisson(x*lam)/float(lam))

def scanline_banding(x, s):
    amp = [0.03, 0.06, 0.11, 0.17, 0.25][s-1]; h = x.shape[0]
    period = np.random.uniform(3, 22); phase = np.random.uniform(0, 2*np.pi)
    b = (1 + amp*np.sin(2*np.pi*np.arange(h)/period + phase)).astype(np.float32)
    return _clip(x * (b[:, None, None] if x.ndim == 3 else b[:, None]))

# ------------------------------------------------------------- photometric --
def illumination_gradient(x, s):
    st = [0.10, 0.20, 0.32, 0.46, 0.62][s-1]; h, w = x.shape[:2]
    ang = np.random.uniform(0, 2*np.pi); yy, xx = np.mgrid[0:h, 0:w]
    u = (xx/w-.5)*np.cos(ang) + (yy/h-.5)*np.sin(ang)
    g = (1 + st*u/(np.abs(u).max()+1e-8)).astype(np.float32)
    return _clip(x * _mask3(g, x))

def brightness_drift(x, s):
    # v2: gamma instead of an additive offset. Gamma maps [0,1] -> [0,1] and CANNOT clip.
    # v1 saturated 36.8% of pixels at severity 5 on Magnetic Tile, which destroyed dynamic
    # range rather than shifting brightness and made the top of the ladder meaningless.
    g = [1.12, 1.26, 1.45, 1.72, 2.05][s-1]
    if np.random.rand() < 0.5: g = 1.0/g
    return _clip(np.power(_clip(x), g))

def contrast_loss(x, s):
    g = [0.80, 0.65, 0.50, 0.36, 0.24][s-1]
    m = x.mean(axis=(0, 1), keepdims=True)
    return _clip((x-m)*g + m)

# ---------------------------------------------------------------- pipeline --
def jpeg_compression(x, s):
    q = [70, 50, 32, 18, 9][s-1]
    src = (x[..., ::-1]*255).astype(np.uint8) if x.ndim == 3 else (x*255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", src, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR if x.ndim == 3 else cv2.IMREAD_GRAYSCALE)
    return (dec[..., ::-1] if x.ndim == 3 else dec).astype(np.float32)/255.

CORRUPTIONS = {
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "vignetting": vignetting,
    "lens_contamination": lens_contamination, "vibration_jitter": vibration_jitter,
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise,
    "scanline_banding": scanline_banding, "illumination_gradient": illumination_gradient,
    "brightness_drift": brightness_drift, "contrast_loss": contrast_loss,
    "jpeg": jpeg_compression,
}
TRAIN_FAMILIES = ["defocus_blur", "gaussian_noise", "illumination_gradient",
                  "jpeg", "lens_contamination", "scanline_banding"]
TEST_FAMILIES  = ["motion_blur", "shot_noise", "brightness_drift",
                  "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]
CONDITIONS = [("clean", 0)] + [(f, s) for f in CORRUPTIONS for s in SEVERITIES]


def corruption_seed(image_id, family, severity=None):
    # Depends on (image_id, family) only, NOT severity: nuisance parameters (blur angle,
    # banding period, blob positions, gradient orientation) stay fixed so that severity is
    # the sole varying factor. Seeding on severity too made scanline_banding score 0.20.
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return int.from_bytes(h[:4], "little")


def apply_corruption(img_u8, family, severity, image_id=None, seed=None):
    if family == "clean":
        return img_u8
    if seed is None and image_id is not None:
        seed = corruption_seed(image_id, family)
    st = None
    if seed is not None:
        st = np.random.get_state(); np.random.seed(seed % (2**32))
    try:
        out = CORRUPTIONS[family](img_u8.astype(np.float32)/255., severity)
    finally:
        if st is not None: np.random.set_state(st)
    return (np.clip(out, 0, 1)*255).astype(np.uint8)


# ------------------------------------------------------- quality descriptors -
def _gray(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im
    return g.astype(np.float32)/255.

def _sharp(im): return float(cv2.Laplacian(_gray(im), cv2.CV_32F).var())

def _noise(im):
    g = _gray(im); h, w = g.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.abs(cv2.filter2D(g, -1, M)).sum()*math.sqrt(math.pi/2) /
                 (6*max(w-2, 1)*max(h-2, 1)))

def _block(im):
    dh = np.abs(np.diff(_gray(im), axis=1))
    on = dh[:, 7::8].mean() if dh.shape[1] > 8 else 0.0
    return float(on/(dh.mean()+1e-8) - 1.0)

def _hf(im):
    g = _gray(im); f = np.abs(np.fft.fftshift(np.fft.fft2(g))); h, w = g.shape
    cy, cx = h//2, w//2; r = max(4, min(h, w)//8)
    return float(1.0 - f[cy-r:cy+r, cx-r:cx+r].sum()/(f.sum()+1e-8))


def quality_descriptor(img_u8):
    # 8-d ABSOLUTE no-reference descriptor (unchanged from v1).
    g = _gray(img_u8); h, w = g.shape
    if img_u8.ndim == 3:
        rg = img_u8[..., 0].astype(np.float32) - img_u8[..., 1]
        yb = .5*(img_u8[..., 0].astype(np.float32) + img_u8[..., 1]) - img_u8[..., 2]
        colour = float((np.sqrt(rg.std()**2 + yb.std()**2)
                        + .3*np.sqrt(rg.mean()**2 + yb.mean()**2))/255.)
    else:
        colour = 0.0
    cen = g[h//4:3*h//4, w//4:3*w//4]
    per = (g.sum()-cen.sum())/max(g.size-cen.size, 1)
    v = np.array([math.log1p(max(_sharp(img_u8), 0.)*1e3), _hf(img_u8),
                  math.log1p(max(_noise(img_u8), 0.)*1e3), _block(img_u8),
                  float(g.mean()), float(g.std()), colour,
                  float(cen.mean()/(per+1e-8))], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


def quality_descriptor_relative(img_u8):
    # 8-d PERTURBATION-RESPONSE descriptor: how much does a statistic move when a known
    # perturbation is applied? Higher severity signal than the absolute set (0.324 vs 0.236
    # on the synthetic benchmark) but NO reduction in class leakage on its own -- ratios
    # remove the absolute texture level, not the spectral shape. Use with class-conditional
    # standardisation, never alone.
    g8 = (_gray(img_u8)*255).astype(np.uint8); eps = 1e-8
    b = cv2.GaussianBlur(g8, (0, 0), 1.5)
    dn = cv2.resize(cv2.resize(g8, (max(2, g8.shape[1]//2), max(2, g8.shape[0]//2)),
                               interpolation=cv2.INTER_AREA),
                    (g8.shape[1], g8.shape[0]), interpolation=cv2.INTER_LINEAR)
    rn = np.random.default_rng(12345)
    nz = np.clip(g8.astype(np.float32) + rn.normal(0, 12, g8.shape), 0, 255).astype(np.uint8)
    _, e = cv2.imencode(".jpg", g8, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
    jp = cv2.imdecode(e, cv2.IMREAD_GRAYSCALE)
    kh = np.zeros((9, 9), np.float32); kh[4, :] = 1/9.
    hb = cv2.filter2D(g8.astype(np.float32), -1, kh).astype(np.uint8)
    vb = cv2.filter2D(g8.astype(np.float32), -1, kh.T).astype(np.uint8)
    x = g8.astype(np.float32)/255.
    v = np.array([
        math.log((_sharp(g8)+eps)/(_sharp(b)+eps)),
        math.log((_sharp(g8)+eps)/(_sharp(dn)+eps)),
        math.log((_noise(nz)+eps)/(_noise(g8)+eps)),
        _block(jp) - _block(g8),
        float(((x+0.25) > 1.0).mean() + ((x-0.25) < 0.0).mean()),
        abs(math.log((_sharp(hb)+eps)/(_sharp(vb)+eps))),
        math.log((_hf(g8)+eps)/(_hf(b)+eps)),
        math.log((_gray(g8).std()+eps)/(_gray(b).std()+eps)),
    ], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


QUALITY_NAMES = ["sharpness", "hf_energy", "noise", "blockiness",
                 "luminance_mean", "luminance_std", "colourfulness", "vignette_ratio"]
RELATIVE_NAMES = ["blur_headroom", "resolution_headroom", "noise_headroom",
                  "compression_headroom", "clipping_headroom", "blur_anisotropy",
                  "hf_retention", "contrast_retention"]
QUALITY_DIM = 8


class ClassConditionalStandardiser:
    # z = (q - mu_c) / sigma_c, with mu_c and sigma_c estimated on DEVELOPMENT data only.
    #
    # Rationale: degradation is relative. A blurry-looking crazing image and a sharp-looking
    # patches image can have identical absolute sharpness; what makes one degraded is that it
    # is blurrier than crazing images normally are. Removing the class-conditional mean strips
    # the content component and leaves the deviation-from-typical -- which is the degradation.
    #
    # At test time c is the model's own prediction. There is no feedback loop: a per-image
    # scalar temperature cannot change the argmax, so the prediction is fixed before the
    # calibrator runs.
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.mu = None; self.sd = None

    def fit(self, q, y):
        d = q.shape[1]
        self.mu = np.zeros((self.n_classes, d), np.float32)
        self.sd = np.ones((self.n_classes, d), np.float32)
        gm, gs = q.mean(0), q.std(0) + 1e-6
        for c in range(self.n_classes):
            m = (y == c)
            if m.sum() >= 5:                 # fall back to global stats for tiny classes
                self.mu[c] = q[m].mean(0); self.sd[c] = q[m].std(0) + 1e-6
            else:
                self.mu[c] = gm; self.sd[c] = gs
        return self

    def transform(self, q, y_pred):
        y_pred = np.asarray(y_pred).astype(int)
        return ((q - self.mu[y_pred]) / self.sd[y_pred]).astype(np.float32)
"""
print(f"embedded core module: {len(CORE_V2.splitlines())} lines")

embedded core module: 246 lines


## 1. Setup and feature extraction

Same seed, subset, backbone and conditions as Phase 0.6/0.7 so numbers remain comparable.

In [22]:
#@title Dependencies and core module
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11","scikit-learn==1.5.2",
                "opencv-python-headless==4.10.0.84","pandas==2.2.3","kagglehub","tqdm"],check=True)

import os, json, math, time, random, hashlib, warnings
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")

SEED=20260821
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
ROOT=Path("/content/sdic"); OUT=ROOT/"phase0"; OUT.mkdir(parents=True,exist_ok=True)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

sys.path.insert(0,str(ROOT))
_t=ROOT/"sdic_core_v2.py"; _w=hashlib.sha256(CORE_V2.encode()).hexdigest()
if not _t.exists() or hashlib.sha256(_t.read_text().encode()).hexdigest()!=_w:
    _t.write_text(CORE_V2); print(f"sdic_core_v2.py written, sha256 {_w[:16]}")
else: print(f"sdic_core_v2.py matches audited copy, sha256 {_w[:16]}")
import importlib, sdic_core_v2; importlib.reload(sdic_core_v2)
from sdic_core_v2 import (CORRUPTIONS, TRAIN_FAMILIES, TEST_FAMILIES, SEVERITIES,
                          apply_corruption, quality_descriptor, QUALITY_DIM)
print("device:", DEVICE)

sdic_core_v2.py matches audited copy, sha256 fd8e3c534562f9d2
device: cuda


In [3]:
#@title Index NEU and extract features (the only expensive cell)
import kagglehub
from timm.data import resolve_model_data_config
_INTERP={"bilinear":cv2.INTER_LINEAR,"bicubic":cv2.INTER_CUBIC,
         "nearest":cv2.INTER_NEAREST,"area":cv2.INTER_AREA}
NEU_ROOT=Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
neu=[p for p in sorted(NEU_ROOT.rglob("*.jpg"))
     if p.parent.name.lower() not in {"train","validation","images","annotations"}]
labels=np.array([p.parent.name for p in neu])
LUT={n:i for i,n in enumerate(sorted(set(labels)))}
y_all=np.array([LUT[l] for l in labels]); NC=len(LUT)
def rd(p): return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)

BACKBONE="resnet50.a1_in1k"; N_PER_CLASS=150
set_seed()
per=defaultdict(list)
for p,yy in zip(neu,y_all): per[yy].append(p)
sel=[]
for c,v in per.items():
    pick=np.random.default_rng(SEED+c).choice(len(v),min(N_PER_CLASS,len(v)),replace=False)
    sel += [neu.index(v[t]) for t in pick]
sel=np.array(sorted(sel))
# REPLICATION: the block above reproduces the selection of the paper's 900 images (ORIG).
# This run uses the images of each class that the paper did NOT use, so that no image is shared.
ORIG_paths=[neu[i] for i in sel]; _orig_idx=set(sel.tolist()); _orig_stems={p.stem for p in ORIG_paths}
sel2=[]
for c,v in sorted(per.items()):
    rest=[neu.index(p) for p in v if neu.index(p) not in _orig_idx and p.stem not in _orig_stems]
    if len(rest)>N_PER_CLASS:
        rest=[rest[t] for t in np.random.default_rng(SEED+100+c).choice(len(rest),N_PER_CLASS,replace=False)]
    print(f"class {c}: {len(v)} images in the dataset, {len(rest)} unused ones taken")
    sel2+=rest
sel=np.array(sorted(sel2)); S_paths=[neu[i] for i in sel]; S_y=y_all[sel]
assert not ({p.stem for p in S_paths} & _orig_stems), "replication subset overlaps the original subset"
assert len(set(p.stem for p in S_paths))==len(S_paths), "duplicate file names in the replication subset"
SUBSET_MAP=pd.DataFrame(dict(image_id=np.arange(len(S_paths)),file=[p.name for p in S_paths],
                             label=[p.parent.name for p in S_paths]))
print(f"{len(S_paths)} images, {NC} classes; 0 shared with the {len(ORIG_paths)} images of the paper")

model=timm.create_model(BACKBONE,pretrained=True,num_classes=0).eval().to(DEVICE)
_c=resolve_model_data_config(model)
PP={"mean":np.array(_c["mean"],np.float32),"std":np.array(_c["std"],np.float32),"size":224,
    "interp":_INTERP.get(_c["interpolation"],cv2.INTER_CUBIC),
    "crop_pct":float(_c.get("crop_pct") or 1.0)}
def prep(im):
    sz=PP["size"]; to=int(round(sz/PP["crop_pct"])); h,w=im.shape[:2]; s=to/min(h,w)
    r=cv2.resize(im,(max(1,int(round(w*s))),max(1,int(round(h*s)))),interpolation=PP["interp"])
    hh,ww=r.shape[:2]; t,l=(hh-sz)//2,(ww-sz)//2
    x=(r[t:t+sz,l:l+sz].astype(np.float32)/255.-PP["mean"])/PP["std"]
    return torch.from_numpy(x).permute(2,0,1)

# FIX: Ensure all severities are cached for TRAIN_FAMILIES
CONDS=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in SEVERITIES] \
                   +[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]

@torch.no_grad()
def extract(fam,sev,bs=96):
    Fs,Qs=[],[]
    for i in range(0,len(S_paths),bs):
        xs,qs=[],[]
        for p in S_paths[i:i+bs]:
            im=apply_corruption(rd(p),fam,max(sev,1),image_id=p.stem) if fam!="clean" else rd(p)
            qs.append(quality_descriptor(im)); xs.append(prep(im))
        with torch.autocast("cuda",enabled=DEVICE=="cuda"):
            Fs.append(model(torch.stack(xs).to(DEVICE)).float().cpu().numpy())
        Qs.append(np.stack(qs))
    return np.concatenate(Fs),np.concatenate(Qs)

t0=time.time(); CACHE={}
for fam,sev in tqdm(CONDS,desc="extract"): CACHE[(fam,sev)]=extract(fam,sev)
del model; torch.cuda.empty_cache()
print(f"{len(CONDS)} conditions in {(time.time()-t0)/60:.1f} min")
print("Every image is now cached under every condition, so all five folds are free.")


Using Colab cache for faster access to the 'neu-surface-defect-database' dataset.
class 0: 300 images in the dataset, 150 unused ones taken
class 1: 300 images in the dataset, 150 unused ones taken
class 2: 300 images in the dataset, 150 unused ones taken
class 3: 300 images in the dataset, 150 unused ones taken
class 4: 300 images in the dataset, 150 unused ones taken
class 5: 300 images in the dataset, 150 unused ones taken
900 images, 6 classes; 0 shared with the 900 images of the paper


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

extract:   0%|          | 0/61 [00:00<?, ?it/s]

61 conditions in 9.3 min
Every image is now cached under every condition, so all five folds are free.


## 2. Standardiser, temperature heads, metrics

In [4]:
class RobustClassStandardiser:
    """z = clip((q - mu_c)/sigma_c, +/- clip_z). The variance floor is what stopped the
    collapse: without it a near-constant class-dimension produced |z| in the hundreds of
    thousands, and fixing only the fit distribution was not enough (C6b still collapsed)."""
    def __init__(self,n_classes,sd_floor=0.25,clip_z=5.0):
        self.n=n_classes; self.sd_floor=sd_floor; self.clip_z=clip_z
    def fit(self,q,y):
        gm,gs=q.mean(0),q.std(0)+1e-6
        self.mu=np.tile(gm,(self.n,1)).astype(np.float32)
        self.sd=np.tile(gs,(self.n,1)).astype(np.float32)
        for c in range(self.n):
            m=(y==c)
            if m.sum()>=20:
                self.mu[c]=q[m].mean(0)
                self.sd[c]=np.maximum(q[m].std(0),self.sd_floor*gs)+1e-6
        return self
    def transform(self,q,y_pred):
        y_pred=np.asarray(y_pred).astype(int)
        return np.clip((q-self.mu[y_pred])/self.sd[y_pred],
                       -self.clip_z,self.clip_z).astype(np.float32)

EPS=1e-2
class ScalarT(nn.Module):
    def __init__(self,d=None):
        super().__init__(); self.log_t=nn.Parameter(torch.zeros(()))
    def temperature(self,q): return self.log_t.exp().expand(q.shape[0])+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class LinearT(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        nn.init.constant_(self.lin.bias,math.log(math.exp(1.0-EPS)-1.0))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q): return F.softplus(self.lin((q-self.mu)/self.sd).squeeze(-1))+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class BoundedT(nn.Module):
    """T = T_lo + (T_hi - T_lo)*sigmoid(w.z + b): collapse is structurally impossible."""
    def __init__(self,d,t_lo=0.5,t_hi=5.0):
        super().__init__(); self.t_lo,self.t_hi=t_lo,t_hi
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        p=(1.0-t_lo)/(t_hi-t_lo); nn.init.constant_(self.lin.bias,math.log(p/(1-p)))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q):
        return self.t_lo+(self.t_hi-self.t_lo)*torch.sigmoid(
            self.lin((q-self.mu)/self.sd).squeeze(-1))
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class MLPT(LinearT):
    def __init__(self,d,h=32):
        super().__init__(d); self.lin=None
        self.net=nn.Sequential(nn.Linear(d,h),nn.SiLU(),nn.Linear(h,h//2),nn.SiLU(),
                               nn.Linear(h//2,1))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias,math.log(math.exp(1.0-EPS)-1.0))
    def temperature(self,q): return F.softplus(self.net((q-self.mu)/self.sd).squeeze(-1))+EPS

def fit_cal(cls,L,Q,Y,epochs=400,lr=1e-2,seed=SEED):
    set_seed(seed)
    L=torch.as_tensor(L,dtype=torch.float32); Q=torch.as_tensor(Q,dtype=torch.float32)
    Y=torch.as_tensor(Y,dtype=torch.long)
    m=cls(Q.shape[1])
    if hasattr(m,"fit_norm"): m.fit_norm(Q)
    opt=torch.optim.Adam(m.parameters(),lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(m(L,Q),Y).backward(); opt.step()
    return m.eval()

def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def ece_em(p,y,nb=15):
    conf,pred=p.max(1),p.argmax(1); corr=(pred==y).astype(float); o=np.argsort(conf)
    return float(sum(len(c)/len(y)*abs(corr[c].mean()-conf[c].mean())
                     for c in np.array_split(o,nb) if len(c)))
def nll_pi(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None))
def aurc(p,y):
    conf,pred=p.max(1),p.argmax(1); e=(pred!=y).astype(float)[np.argsort(-conf)]
    n=len(e); return float(np.trapezoid(np.cumsum(e)/np.arange(1,n+1),np.arange(1,n+1)/n))

for cls in (LinearT,BoundedT,MLPT):
    m=cls(8); m.fit_norm(torch.randn(200,8))
    T=m.temperature(torch.randn(200,8)).detach()
    assert abs(T.mean().item()-1.0)<1e-4 and T.std().item()<1e-5, cls.__name__
print("all heads initialise to T = 1.0 exactly")

all heads initialise to T = 1.0 exactly


## 3. Five folds, with selection on a held-out split

Per fold: 60% probe training and standardiser reference, 20% calibrator fitting, 20% **selection**,
20% test. The selection split shares no images with either the calibrator fit or the test fold,
and the test fold is read exactly once, at the end.

Selection is run twice — under train families (strict, the primary) and under test families
(optimistic, a sensitivity check) — so any disagreement between them is visible rather than
hidden.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split

# NESTED FAMILY HOLDOUT. The six train families are split: four fit the calibrators, two are
# held out for selection. Selection therefore measures generalisation to UNSEEN FAMILIES on
# UNSEEN IMAGES -- the same kind of shift the test set measures -- without ever touching the
# six reported test families.
CAL_FAMILIES = TRAIN_FAMILIES[:4]
SEL_FAMILIES = TRAIN_FAMILIES[4:]
CAL_CONDS       =[("clean",0)]+[(f,s) for f in CAL_FAMILIES for s in (1,3,5)]
SEL_CONDS_NESTED=[(f,s) for f in SEL_FAMILIES  for s in SEVERITIES]   # primary
SEL_CONDS_STRICT=[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)]    # v0.9 rule, kept to expose it
SEL_CONDS_OPTIM =[(f,s) for f in TEST_FAMILIES  for s in SEVERITIES] # compromised, diagnostic only
TEST_CONDS      =[(f,s) for f in TEST_FAMILIES  for s in SEVERITIES]
print(f"calibration families {CAL_FAMILIES}\nselection families   {SEL_FAMILIES}\n"
      f"test families        {TEST_FAMILIES}")

ARM_SPECS={
 "C2 scalar / augmented":      ("scalar",ScalarT, "q"),
 "C1 scalar / clean":          ("clean", ScalarT, "q"),
 "C3 MLP / raw q":             ("scalar",MLPT,    "q"),
 "C4 linear / raw q":          ("scalar",LinearT, "q"),
 "C5 linear / class one-hot":  ("scalar",LinearT, "h"),
 "C6c linear / std (robust)":  ("scalar",LinearT, "z"),
 "C7 bounded / std (robust)":  ("scalar",BoundedT,"z"),
 "C7q bounded / raw q":        ("scalar",BoundedT,"q"),
 "C8 linear / class + q":      ("scalar",LinearT, "hq"),
 "C8b bounded / class + q":    ("scalar",BoundedT,"hq"),
}
QUALITY_ARMS=[k for k in ARM_SPECS if k.startswith(("C4","C6","C7","C8"))]

def gather(idx,conds,logit_fn,std=None):
    L,Q,Y,IM=[],[],[],[]
    for fam,sev in conds:
        Fx,Qx=CACHE[(fam,sev)]
        L.append(logit_fn(Fx[idx])); Q.append(Qx[idx]); Y.append(S_y[idx]); IM.append(idx)
    L=np.concatenate(L); Q=np.concatenate(Q); Y=np.concatenate(Y); IM=np.concatenate(IM)
    H=np.zeros((len(L),NC),np.float32); H[np.arange(len(L)),L.argmax(1)]=1.0
    Z=std.transform(Q,L.argmax(1)) if std is not None else None
    HQ=np.concatenate([H,Q],1).astype(np.float32)      # C8: class offset + quality slope
    return dict(L=L,q=Q,h=H,z=Z,hq=HQ,y=Y,im=IM)

def evaluate(m,pk,feat):
    X=torch.as_tensor(pk[feat],dtype=torch.float32)
    with torch.no_grad():
        out=m(torch.as_tensor(pk["L"],dtype=torch.float32),X).numpy()
        T=m.temperature(X).numpy()
    p=softmax(out)
    return dict(nll=float(nll_pi(p,pk["y"]).mean()),ece=ece_em(p,pk["y"]),
                aurc=aurc(p,pk["y"]),acc=float((p.argmax(1)==pk["y"]).mean()),
                T_min=float(T.min()),T_max=float(T.max())), p, nll_pi(p,pk["y"])

def sound(r):
    return not ((r["T_min"]-EPS<0.1*EPS and r["T_min"]<1.0) or r["T_max"]>50)

skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
FOLDS=[te for _,te in skf.split(np.zeros(len(S_y)),S_y)]
Fc,Qc=CACHE[("clean",0)]

fold_rows=[]; sel_rows=[]; fam_rows=[]
POOL =defaultdict(lambda: dict(nll=[],y=[],im=[],p=[]))
POOLF=defaultdict(lambda: dict(nll=[],y=[],im=[],p=[]))
for k in range(5):
    te=FOLDS[k]; rest=np.concatenate([FOLDS[j] for j in range(5) if j!=k])
    tr,tmp=train_test_split(rest,test_size=0.4,random_state=SEED+k,stratify=S_y[rest])
    va_fit,va_sel=train_test_split(tmp,test_size=0.5,random_state=SEED+k,stratify=S_y[tmp])

    probe=make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=4000,class_weight="balanced"))
    probe.fit(Fc[tr],S_y[tr])
    def lg_of(Fx,_p=probe):
        d=_p.decision_function(Fx); return d if d.ndim>1 else np.stack([-d,d],1)

    dev=gather(tr,CAL_CONDS,lg_of)
    STD=RobustClassStandardiser(NC).fit(dev["q"],dev["y"])
    P_fit  =gather(va_fit,CAL_CONDS,lg_of,STD)
    P_fitc =gather(va_fit,[("clean",0)],lg_of,STD)
    P_selN =gather(va_sel,SEL_CONDS_NESTED,lg_of,STD)
    P_selS =gather(va_sel,SEL_CONDS_STRICT,lg_of,STD)
    P_selO =gather(va_sel,SEL_CONDS_OPTIM ,lg_of,STD)
    P_te   =gather(te,TEST_CONDS,lg_of,STD)
    P_te_by_fam={f:gather(te,[(f,sv) for sv in SEVERITIES],lg_of,STD) for f in TEST_FAMILIES}

    for name,(mode,cls,feat) in ARM_SPECS.items():
        src = P_fitc if mode=="clean" else P_fit
        m = fit_cal(cls, src["L"], src[feat], src["y"])
        rte,pte,pim = evaluate(m,P_te,feat)
        rsn,_,_ = evaluate(m,P_selN,feat)
        rss,_,_ = evaluate(m,P_selS,feat)
        rso,_,_ = evaluate(m,P_selO,feat)
        fold_rows.append(dict(fold=k,arm=name,**rte,sound=sound(rte)))
        sel_rows.append(dict(fold=k,arm=name,
                             sel_nested_nll=rsn["nll"],sel_nested_ece=rsn["ece"],
                             sel_strict_nll=rss["nll"],sel_strict_ece=rss["ece"],
                             sel_optim_nll=rso["nll"],sel_optim_ece=rso["ece"],
                             sel_sound=sound(rsn) and sound(rss)))
        POOL[name]["nll"].append(pim); POOL[name]["y"].append(P_te["y"])
        POOL[name]["im"].append(P_te["im"]); POOL[name]["p"].append(pte)
        for f,pk in P_te_by_fam.items():
            rf,pf,pimf = evaluate(m,pk,feat)
            fam_rows.append(dict(fold=k,arm=name,family=f,**rf))
            POOLF[(name,f)]["nll"].append(pimf); POOLF[(name,f)]["y"].append(pk["y"])
            POOLF[(name,f)]["im"].append(pk["im"]); POOLF[(name,f)]["p"].append(pf)

    p0=softmax(P_te["L"])
    fold_rows.append(dict(fold=k,arm="C0 uncalibrated",nll=float(nll_pi(p0,P_te["y"]).mean()),
                          ece=ece_em(p0,P_te["y"]),aurc=aurc(p0,P_te["y"]),
                          acc=float((p0.argmax(1)==P_te["y"]).mean()),T_min=1.,T_max=1.,sound=True))
    POOL["C0 uncalibrated"]["nll"].append(nll_pi(p0,P_te["y"]))
    POOL["C0 uncalibrated"]["y"].append(P_te["y"]); POOL["C0 uncalibrated"]["im"].append(P_te["im"])
    POOL["C0 uncalibrated"]["p"].append(p0)
    print(f"  fold {k} done")

FR=pd.DataFrame(fold_rows); SR=pd.DataFrame(sel_rows); FAM=pd.DataFrame(fam_rows)
for d in list(POOL.values())+list(POOLF.values()):
    for kk in ("nll","y","im","p"): d[kk]=np.concatenate(d[kk])
print(f"\n{len(FR)} arm-fold results | pooled test rows per arm: {len(POOL['C4 linear / raw q']['y'])}")

calibration families ['defocus_blur', 'gaussian_noise', 'illumination_gradient', 'jpeg']
selection families   ['lens_contamination', 'scanline_banding']
test families        ['motion_blur', 'shot_noise', 'brightness_drift', 'contrast_loss', 'vignetting', 'vibration_jitter']
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done

55 arm-fold results | pooled test rows per arm: 27000


## 4. Which arm does validation choose?

In [6]:
UNCAL=FR[FR.arm=="C0 uncalibrated"].nll.mean()
agg=(FR.groupby("arm").agg(nll_mean=("nll","mean"),nll_sd=("nll","std"),
                           ece_mean=("ece","mean"),ece_sd=("ece","std"),
                           aurc_mean=("aurc","mean"),T_min=("T_min","min"),
                           n_sound=("sound","sum")).round(4))
# C0 is the reference; comparing it against itself produced a rounding artefact in v0.8
# (agg.nll_mean is rounded to 4dp, UNCAL is not, so 1.1255 > 1.125494 came out True).
agg["gate"]=np.where(agg.index=="C0 uncalibrated","reference",
             np.where(agg.n_sound<5,"FAILS in >=1 fold",
             np.where(agg.nll_mean>UNCAL+1e-6,"worse than uncalibrated","sound")))
display(agg.sort_values("nll_mean"))
agg.to_csv(OUT/"phase0_11_arms.csv")

sa=SR.groupby("arm").agg(nested_nll=("sel_nested_nll","mean"),nested_ece=("sel_nested_ece","mean"),
                         strict_nll=("sel_strict_nll","mean"),strict_ece=("sel_strict_ece","mean"),
                         optim_nll=("sel_optim_nll","mean"),optim_ece=("sel_optim_ece","mean"),
                         n_sound=("sel_sound","sum")).round(4)
elig=[a for a in QUALITY_ARMS if a in sa.index and sa.loc[a,"n_sound"]==5
      and agg.loc[a,"gate"]=="sound"]
print("\nSELECTION (test fold never consulted)")
display(sa.loc[[a for a in QUALITY_ARMS]])
from scipy.stats import spearmanr
if elig:
    W_NLL = min(elig,key=lambda a: sa.loc[a,"nested_nll"])
    W_ECE = min(elig,key=lambda a: sa.loc[a,"nested_ece"])
    print(f"\n  winner on NESTED validation NLL : {W_NLL}")
    print(f"  winner on NESTED validation ECE : {W_ECE}")
    if W_NLL!=W_ECE:
        print("  the two metrics select different arms; both are carried forward")
    WINNERS=sorted({W_NLL,W_ECE})

    # Does each selection rule actually predict test performance?
    print("\n  Does the selection signal predict the test outcome? (Spearman over eligible arms)")
    for rule,col in [("nested (4 fit / 2 held-out families)","nested_nll"),
                     ("strict  (all train families)","strict_nll"),
                     ("optimistic (test families)","optim_nll")]:
        rho=spearmanr([sa.loc[a,col] for a in elig],
                      [agg.loc[a,"nll_mean"] for a in elig]).statistic
        flag="" if rho>=0.6 else "   <- WEAK: this rule does not predict the test outcome"
        print(f"    {rule:38s} rho = {rho:+.3f}{flag}")
    RHO_BEST=max(spearmanr([sa.loc[a,c] for a in elig],
                           [agg.loc[a,"nll_mean"] for a in elig]).statistic
                 for c in ("nested_nll","strict_nll"))
    print(f"  best legitimate rule: rho = {RHO_BEST:+.3f}"
          f"  ({'usable' if RHO_BEST>=0.6 else 'NOT usable to choose the method'})")
    print("  (the optimistic rule is excluded here: it selects on the reported test families)")
else:
    WINNERS=[]; RHO_BEST=float("nan")
    print("\nNo quality-conditioned arm is sound in all five folds.")

,nll_mean,nll_sd,ece_mean,ece_sd,aurc_mean,T_min,n_sound,gate
arm,,,,,,,,
C2 scalar / augmented,0.7559,0.0359,0.0219,0.0089,0.0791,1.8502,5,sound
C7q bounded / raw q,0.8514,0.0675,0.0394,0.0094,0.0693,0.5082,5,sound
C7 bounded / std (robust),0.8694,0.0606,0.0402,0.0067,0.0704,0.5150,5,sound
C5 linear / class one-hot,0.8915,0.1541,0.0422,0.0143,0.0847,0.3271,5,sound
C4 linear / raw q,0.9532,0.1905,0.0415,0.0132,0.0796,0.0185,5,sound
C8b bounded / class + q,0.9739,0.0858,0.0543,0.0085,0.0745,0.5014,5,sound
C6c linear / std (robust),1.0229,0.1352,0.0442,0.0096,0.0876,0.0337,5,sound
C0 uncalibrated,1.0640,0.1012,0.1194,0.0077,0.0758,1.0000,5,reference
C8 linear / class + q,1.2005,0.2692,0.0609,0.0128,0.0878,0.0103,4,FAILS in >=1 fold



SELECTION (test fold never consulted)


,nested_nll,nested_ece,strict_nll,strict_ece,optim_nll,optim_ece,n_sound
arm,,,,,,,
C4 linear / raw q,2.0300,0.1854,1.0653,0.0699,0.9526,0.0442,5
C6c linear / std (robust),1.9397,0.1812,1.0323,0.0670,1.0373,0.0476,5
C7 bounded / std (robust),1.7568,0.1790,0.9638,0.0662,0.8732,0.0416,5
C7q bounded / raw q,1.7730,0.1807,0.9683,0.0679,0.8526,0.0396,5
C8 linear / class + q,2.0583,0.1797,1.0651,0.0714,1.1933,0.0615,5
C8b bounded / class + q,1.7832,0.1755,0.9589,0.0684,0.9740,0.0535,5



  winner on NESTED validation NLL : C7 bounded / std (robust)
  winner on NESTED validation ECE : C8b bounded / class + q
  the two metrics select different arms; both are carried forward

  Does the selection signal predict the test outcome? (Spearman over eligible arms)
    nested (4 fit / 2 held-out families)   rho = +0.600
    strict  (all train families)           rho = +0.100   <- WEAK: this rule does not predict the test outcome
    optimistic (test families)             rho = +1.000
  best legitimate rule: rho = +0.600  (usable)
  (the optimistic rule is excluded here: it selects on the reported test families)


## 5. Two kinds of uncertainty, and only one of them was being reported

Every interval so far resampled **images inside a fixed set of six test families**. That answers
*"would this hold on different images of these six corruptions?"* It does not answer *"would this
hold on six different corruptions?"* — which is the question the paper actually asks.

The Phase 0.10 run showed those are wildly different questions. The two held-out selection
families produced NLL around 2.2–2.4 while the six test families produced 0.76–1.03: roughly
2.5× harder. Corruption families are **not exchangeable**, so treating six of them as a large
sample is wrong. The effective sample size for a claim about unseen corruption is closer to
**n = 6** than to n = 5,400.

Both intervals are therefore computed for every comparison:

* **image-level** — resamples images, conditional on this family set. Narrow. Reported so far.
* **family-level** — computes the per-family difference, then a *t* interval over the six
  families with 5 degrees of freedom. This is the one that supports a generalisation claim.

A method that wins on the image interval but not the family interval has been shown to work on
*these six corruptions*, not on corruption in general. That distinction has to reach the paper.

In [7]:
from scipy.stats import t as _t

def image_boot_nll(a,b,ims,n_boot=10000,seed=SEED):
    rng=np.random.default_rng(seed); d=a-b
    uniq=np.unique(ims); by={u:np.where(ims==u)[0] for u in uniq}
    bs=np.array([d[np.concatenate([by[u] for u in rng.choice(uniq,len(uniq),True)])].mean()
                 for _ in range(n_boot)])
    return float(d.mean()),float(np.quantile(bs,.025)),float(np.quantile(bs,.975))

def family_ci(per_family_deltas,alpha=0.05):
    v=np.asarray(per_family_deltas,float); n=len(v)
    m=float(v.mean()); sd=float(v.std(ddof=1))
    h=_t.ppf(1-alpha/2,n-1)*sd/math.sqrt(n)
    return m,m-h,m+h,sd,int(n)

def compare(A,B):
    dn_f=[float(POOLF[(A,f)]["nll"].mean()-POOLF[(B,f)]["nll"].mean()) for f in TEST_FAMILIES]
    de_f=[float(ece_em(POOLF[(A,f)]["p"],POOLF[(A,f)]["y"])
                -ece_em(POOLF[(B,f)]["p"],POOLF[(B,f)]["y"])) for f in TEST_FAMILIES]
    dn,ilo,ihi=image_boot_nll(POOL[A]["nll"],POOL[B]["nll"],POOL[A]["im"])
    fm,flo,fhi,fsd,n=family_ci(dn_f)
    em,elo,ehi,esd,_=family_ci(de_f)
    return dict(arm=A,vs=B,
        d_nll=round(dn,4), nll_image_ci=f"[{ilo:.4f}, {ihi:.4f}]", win_nll_image=bool(ihi<0),
        nll_family_ci=f"[{flo:.4f}, {fhi:.4f}]", win_nll_family=bool(fhi<0),
        nll_family_sd=round(fsd,4),
        d_ece=round(em,4), ece_family_ci=f"[{elo:.4f}, {ehi:.4f}]", win_ece_family=bool(ehi<0),
        ece_family_sd=round(esd,4),
        image_ci_width=round(ihi-ilo,5), family_ci_width=round(fhi-flo,5),
        per_family_nll=[round(x,3) for x in dn_f])

REF=["C2 scalar / augmented","C5 linear / class one-hot"]
SOUND_Q=[a for a in QUALITY_ARMS if agg.loc[a,"gate"]=="sound"]
T=pd.DataFrame([compare(A,B) for A in SOUND_Q for B in REF])
display(T[["arm","vs","d_nll","nll_image_ci","win_nll_image","nll_family_ci","win_nll_family",
           "image_ci_width","family_ci_width"]])
T.to_csv(OUT/"phase0_11_bootstrap.csv",index=False)

wr=(T.family_ci_width/T.image_ci_width).replace([np.inf,-np.inf],np.nan).dropna()
print(f"\nfamily interval / image interval width: min {wr.min():.1f}x  "
      f"median {wr.median():.1f}x  max {wr.max():.1f}x")
flips=T[(T.win_nll_image) & (~T.win_nll_family)]
if len(flips):
    print(f"\n{len(flips)} comparison(s) significant on images but NOT across families:")
    for _,r in flips.iterrows():
        print(f"  {r['arm']} vs {r['vs'].split(' ')[0]}: per-family deltas {r['per_family_nll']}")
    print("  These are claims about these six corruptions, not about corruption in general.")

,arm,vs,d_nll,nll_image_ci,win_nll_image,nll_family_ci,win_nll_family,image_ci_width,family_ci_width
0,C4 linear / raw q,C2 scalar / augmented,0.1973,"[0.1691, 0.2257]",False,"[-0.4703, 0.8649]",False,0.05665,1.33524
1,C4 linear / raw q,C5 linear / class one-hot,0.0616,"[0.0289, 0.0942]",False,"[-0.7278, 0.8511]",False,0.06526,1.57889
2,C6c linear / std (robust),C2 scalar / augmented,0.2671,"[0.2358, 0.2987]",False,"[-0.5707, 1.1048]",False,0.06283,1.67548
3,C6c linear / std (robust),C5 linear / class one-hot,0.1314,"[0.0981, 0.1652]",False,"[-0.8213, 1.0841]",False,0.06709,1.90538
4,C7 bounded / std (robust),C2 scalar / augmented,0.1136,"[0.0973, 0.1301]",False,"[-0.3463, 0.5735]",False,0.03278,0.91984
5,C7 bounded / std (robust),C5 linear / class one-hot,-0.0221,"[-0.0494, 0.0044]",False,"[-0.6179, 0.5737]",False,0.05386,1.19156
6,C7q bounded / raw q,C2 scalar / augmented,0.0955,"[0.0791, 0.1119]",False,"[-0.3150, 0.5061]",False,0.03285,0.82111
7,C7q bounded / raw q,C5 linear / class one-hot,-0.0402,"[-0.0675, -0.0143]",True,"[-0.5894, 0.5091]",False,0.05325,1.09854
8,C8b bounded / class + q,C2 scalar / augmented,0.2181,"[0.1882, 0.2478]",False,"[-0.2449, 0.6810]",False,0.05968,0.92595
9,C8b bounded / class + q,C5 linear / class one-hot,0.0824,"[0.0625, 0.1024]",False,"[-0.3887, 0.5534]",False,0.03990,0.94206



family interval / image interval width: min 15.5x  median 23.9x  max 28.4x

1 comparison(s) significant on images but NOT across families:
  C7q bounded / raw q vs C5: per-family deltas [-0.693, -0.113, -0.069, -0.069, 0.913, -0.21]
  These are claims about these six corruptions, not about corruption in general.


In [8]:
print("="*100)
MATRIX=[]
for A in SOUND_Q:
    r2=T[(T.arm==A)&(T.vs==REF[0])].iloc[0]; r5=T[(T.arm==A)&(T.vs==REF[1])].iloc[0]
    MATRIX.append(dict(arm=A,selected=A in WINNERS,
        nll_C2_img=bool(r2.win_nll_image), nll_C2_fam=bool(r2.win_nll_family),
        ece_C2_fam=bool(r2.win_ece_family),
        nll_C5_img=bool(r5.win_nll_image), nll_C5_fam=bool(r5.win_nll_family),
        ece_C5_fam=bool(r5.win_ece_family),
        score_image=int(r2.win_nll_image)+int(r5.win_nll_image),
        score_family=int(r2.win_nll_family)+int(r2.win_ece_family)
                    +int(r5.win_nll_family)+int(r5.win_ece_family)))
M=pd.DataFrame(MATRIX).sort_values(["score_family","score_image"],ascending=False)
display(M); M.to_csv(OUT/"phase0_11_matrix.csv",index=False)

SEL_VALID = (RHO_BEST >= 0.6)
print("="*100)
if not SEL_VALID:
    verdict="UNDETERMINED — the selection rule is invalid"
    guidance=(f"No selection rule ranks arms consistently with the test outcome (best Spearman "
        f"{RHO_BEST:+.3f} against a 0.60 threshold), so no arm can be named 'the method' from "
        f"this evidence. Reading the verdict off whatever the rule happened to pick is how "
        f"Phase 0.6, 0.7, 0.9 and 0.10 each produced a wrong conclusion.\n\n"
        f"The cause is not a patchable bug: corruption families are not exchangeable, so a rule "
        f"tuned on some families cannot rank arms for others. Fix the DESIGN before the full "
        f"experiment -- either enlarge the family pool substantially so held-out families are "
        f"representative, or select on a criterion that does not require family transfer at all "
        f"(for example the simplest arm that is sound in every fold).")
elif M.iloc[0]["score_family"]>=4:
    verdict="METHOD CONTRIBUTION SUPPORTED UNDER FAMILY-LEVEL UNCERTAINTY"
    guidance=(f"{M.iloc[0]['arm']} beats both baselines on both metrics with intervals that "
        f"account for family-to-family variance, not just image variance. That is the strong "
        f"form of the claim and it is what a reviewer will test.")
elif M.iloc[0]["score_family"]>=1:
    verdict="SUPPORTED ON THESE SIX CORRUPTIONS ONLY"
    guidance=(f"{M.iloc[0]['arm']} wins on the image-level intervals but the family-level "
        f"intervals do not all exclude zero. State the scope precisely: the result holds on the "
        f"six corruption families tested, and the evidence does not support generalisation to "
        f"corruption in general. Report the per-family deltas so the reader sees the spread.")
else:
    verdict="NO METHOD CONTRIBUTION UNDER FAMILY-LEVEL UNCERTAINTY"
    guidance=("Once family-to-family variance is accounted for, no arm beats a scalar "
        "temperature fitted on augmented validation data. Reframe around SDI-C, the split "
        "protocol, the leakage precondition and the corrected severity ladders. Report the "
        "negative result explicitly -- it is more useful than an overstated positive one.")
failed=list(agg[~agg.gate.isin(["sound","reference"])].index)
print(f"\nVERDICT: {verdict}\n\n{guidance}")
if failed: print(f"\nExcluded (failed the gate): {failed}")
json.dump({"verdict":verdict,"selection_valid":bool(SEL_VALID),"rho_best":float(RHO_BEST),
           "winners":WINNERS,"guidance":guidance,"excluded":failed,
           "matrix":M.to_dict("records"),"comparisons":T.to_dict("records")},
          open(OUT/"phase0_11_verdict.json","w"),indent=2)

,arm,selected,nll_C2_img,nll_C2_fam,ece_C2_fam,nll_C5_img,nll_C5_fam,ece_C5_fam,score_image,score_family
3,C7q bounded / raw q,False,False,False,False,True,False,False,1,0
0,C4 linear / raw q,False,False,False,False,False,False,False,0,0
1,C6c linear / std (robust),False,False,False,False,False,False,False,0,0
2,C7 bounded / std (robust),True,False,False,False,False,False,False,0,0
4,C8b bounded / class + q,True,False,False,False,False,False,False,0,0



VERDICT: NO METHOD CONTRIBUTION UNDER FAMILY-LEVEL UNCERTAINTY

Once family-to-family variance is accounted for, no arm beats a scalar temperature fitted on augmented validation data. Reframe around SDI-C, the split protocol, the leakage precondition and the corrected severity ladders. Report the negative result explicitly -- it is more useful than an overstated positive one.

Excluded (failed the gate): ['C1 scalar / clean', 'C3 MLP / raw q', 'C8 linear / class + q']


---

## What to carry into the paper

Whatever the verdict, four results are now established on five folds with image-level intervals
and no test-set peeking, and all four belong in the manuscript.

**Clean-fit calibration is worse than no calibration.** Fitted on clean validation data, scalar
temperature scaling drives *T* below 1 and *sharpens* an already-overconfident model. Report the
number; it motivates the entire paper in one line.

**The MLP temperature head fails.** 833 parameters, temperature pinned to the floor, likelihood
worse than uncalibrated. It is ablation evidence against capacity, not a component.

**C5 is the load-bearing control.** Any claim that a temperature tracks image quality rather than
class identity has to survive a calibrator that is handed the predicted class and nothing else.

**The leakage-corrected arm trades likelihood for calibration error.** Report both metrics for
every arm rather than the one that flatters the method.

## Still blocked, and not affected by any of this

KSDD2 from the official ViCoS release. Magnetic Tile folds rebuilt without the discarded pHash
grouping. The F1–F6 fixes applied to the main notebook. And this remains one backbone on one
dataset — the full experiment must show the result replicates across backbones and datasets
before any of it is a paper claim.

---

## 6. Revision: per-image export and the requested analyses

Sections 3 to 5 compute the NLL of every test image under every corruption, severity and arm but keep it only in memory. This section:

1. saves those per-image values (run `G1`, the data behind Tables 3 to 5) and checks them against the section 5 table;
2. refits the five folds with the oracle-class arm C5o added (a calibrator that sees the true class, an upper bound on what class information alone can do) and checks that the refit reproduces section 3 exactly;
3. refits the five folds for further fold partitions (`G1-split1`, `G1-split2`, ...), changing only the seed of the fold split;
4. runs the analyses of `revision_analyses.py` (permutation, Hotelling and studentized tests, leave one family out, fold dependence, per severity, amplitude matching) and shows the report;
5. downloads everything as one zip.

How to read each result is in the *Revision plan* tab of the response document.

In [9]:
#@title 6.1 Settings { display-mode: "form" }
#@markdown Number of fold partitions, the original one included (1 skips the repeated-partition check).
N_PARTITIONS = 10  #@param {type:"integer"}
ADD_ORACLE_ARM = True  #@param {type:"boolean"}
RUN_ANALYSES = True  #@param {type:"boolean"}
#@markdown `QUICK` divides the resampling counts of the analyses by ten for a first pass.
QUICK = False  #@param {type:"boolean"}
DOWNLOAD_ZIP = True  #@param {type:"boolean"}
REV = OUT / "revision"; REV.mkdir(parents=True, exist_ok=True)
print("revision outputs go to", REV)
SUBSET_MAP.to_csv(REV / "replication_subset_files.csv", index=False)
pd.DataFrame(dict(file=[p.name for p in ORIG_paths], label=[p.parent.name for p in ORIG_paths])).to_csv(
    REV / "original_subset_files.csv", index=False)
import json as _json, platform as _pf
_json.dump(dict(python=_pf.python_version(), torch=torch.__version__, timm=timm.__version__, numpy=np.__version__,
                gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"), seed=SEED,
                subset="complement of the paper's 900 NEU-CLS images", n_images=len(S_paths)),
           open(REV / "replication_environment.json", "w"), indent=1)


revision outputs go to /content/sdic/phase0/revision


In [10]:
#@title 6.2 Save the per-image NLLs of sections 3 to 5 (run G1) { display-mode: "form" }
# POOLF holds, per (arm, family), the test images of the five folds in fold order, each fold
# severity-major as gather() builds it; the loop in section 3 concatenated the folds.
_n = [len(f) for f in FOLDS]
_fold = np.concatenate([np.full(len(SEVERITIES) * n, k) for k, n in enumerate(_n)])
_sev = np.concatenate([np.repeat(np.asarray(SEVERITIES), n) for n in _n])
G1_ORIG = pd.concat([pd.DataFrame(dict(
    dataset="NEU-CLS", backbone=BACKBONE, run="G1", fold=_fold, image_id=d["im"], family=fam, severity=_sev,
    arm=arm.split()[0], nll=d["nll"], correct=(d["p"].argmax(1) == d["y"]).astype("int8")))
    for (arm, fam), d in POOLF.items()], ignore_index=True)
print(f"{len(G1_ORIG):,} rows: {G1_ORIG['image_id'].nunique()} images, {G1_ORIG['family'].nunique()} families, "
      f"{G1_ORIG['severity'].nunique()} severities, {G1_ORIG['arm'].nunique()} arms")

# the six per-family differences of every comparison in section 5 must come out of this table
_fm = G1_ORIG.groupby(["arm", "family"])["nll"].mean()
_bad = [f"{r['arm'].split()[0]} vs {r['vs'].split()[0]}" for _, r in T.iterrows()
        if any(abs(float(_fm[(r["arm"].split()[0], f)] - _fm[(r["vs"].split()[0], f)]) - v) > 5e-4 + 1e-9
               for f, v in zip(TEST_FAMILIES, r["per_family_nll"]))]
print(f"per-family differences match the section 5 table for all {len(T)} comparisons" if not _bad
      else f"MISMATCH with the section 5 table: {_bad}")

270,000 rows: 900 images, 6 families, 5 severities, 10 arms
per-family differences match the section 5 table for all 10 comparisons


In [11]:
#@title 6.3 Refit function: oracle arm and fold partitions { display-mode: "form" }
from sklearn.model_selection import StratifiedKFold, train_test_split
ORACLE_ARM = "C5o linear / true-class one-hot"


def gather_x(idx, conds, logit_fn, std=None):
    """gather() plus 'ho', the one-hot of the TRUE class (for the oracle arm only)."""
    pk = gather(idx, conds, logit_fn, std)
    ho = np.zeros((len(pk["y"]), NC), np.float32); ho[np.arange(len(pk["y"])), pk["y"]] = 1.0
    pk["ho"] = ho
    return pk


ARMS_X = dict(ARM_SPECS)
if ADD_ORACLE_ARM:
    ARMS_X[ORACLE_ARM] = ("scalar", LinearT, "ho")


def revision_partition(r, arms=None):
    """Section 3 refitted with the fold split seeded by SEED + 1000 r (r = 0 is the original
    split). Same probe, standardiser, calibrator fits and seeds as section 3; returns one row
    per test image, family (clean included), severity and arm."""
    arms = arms or ARMS_X
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED + 1000 * r)
    folds = [te for _, te in skf.split(np.zeros(len(S_y)), S_y)]
    run = "G1" if r == 0 else f"G1-split{r}"
    out = []
    for k in range(5):
        te = folds[k]; rest = np.concatenate([folds[j] for j in range(5) if j != k])
        tr, tmp = train_test_split(rest, test_size=0.4, random_state=SEED + k, stratify=S_y[rest])
        va_fit, _va_sel = train_test_split(tmp, test_size=0.5, random_state=SEED + k, stratify=S_y[tmp])
        probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000, class_weight="balanced"))
        probe.fit(Fc[tr], S_y[tr])

        def lg_of(Fx, _p=probe):
            d = _p.decision_function(Fx); return d if d.ndim > 1 else np.stack([-d, d], 1)

        dev = gather(tr, CAL_CONDS, lg_of)
        STD = RobustClassStandardiser(NC).fit(dev["q"], dev["y"])
        P_fit = gather_x(va_fit, CAL_CONDS, lg_of, STD)
        P_fitc = gather_x(va_fit, [("clean", 0)], lg_of, STD)
        P_te = {f: gather_x(te, [(f, sv) for sv in SEVERITIES], lg_of, STD) for f in TEST_FAMILIES}
        P_te["clean"] = gather_x(te, [("clean", 0)], lg_of, STD)
        for name, (mode, cls, feat) in arms.items():
            src = P_fitc if mode == "clean" else P_fit
            m = fit_cal(cls, src["L"], src[feat], src["y"])
            for fam, pk in P_te.items():
                _, p, nll = evaluate(m, pk, feat)
                out.append(pd.DataFrame(dict(
                    dataset="NEU-CLS", backbone=BACKBONE, run=run, fold=k, image_id=pk["im"], family=fam,
                    severity=0 if fam == "clean" else np.repeat(np.asarray(SEVERITIES), len(te)),
                    arm=name.split()[0], nll=nll, correct=(pk["L"].argmax(1) == pk["y"]).astype("int8"))))
    return pd.concat(out, ignore_index=True)

In [12]:
#@title 6.4 Refit and save the per-image tables { display-mode: "form" }
TABLES = {}
_t0 = time.time()
for r in tqdm(range(max(1, int(N_PARTITIONS))), desc="fold partitions"):
    TABLES[r] = revision_partition(r)
print(f"{len(TABLES)} partition(s) refitted in {(time.time() - _t0) / 60:.1f} min")

# partition 0 is the original split: the refit must reproduce section 3 for the original arms
_k = ["image_id", "family", "severity", "arm"]
_chk = G1_ORIG[_k + ["nll"]].merge(TABLES[0][_k + ["nll"]], on=_k, suffixes=("_orig", "_refit"))
_dmax = float((_chk["nll_orig"] - _chk["nll_refit"]).abs().max())
REFIT_OK = len(_chk) == len(G1_ORIG) and _dmax < 1e-6
print(f"refit of the original split against section 3: {len(_chk):,} of {len(G1_ORIG):,} rows matched, "
      f"largest NLL difference {_dmax:.1e}: {'identical' if REFIT_OK else 'NOT identical'}")
if not REFIT_OK:
    print("Run G1 therefore keeps the section 3 values for the original arms and takes only C5o from the refit.")
    TABLES[0] = pd.concat([G1_ORIG, TABLES[0][~TABLES[0]["arm"].isin(set(G1_ORIG["arm"]))]], ignore_index=True)

for _tab in TABLES.values():
    _tab.to_csv(REV / f"per_image_nll_{_tab['run'].iloc[0]}.csv.gz", index=False)
print("saved:", ", ".join(sorted(p.name for p in REV.glob("per_image_nll_*.csv.gz"))))

fold partitions:   0%|          | 0/10 [00:00<?, ?it/s]

10 partition(s) refitted in 5.5 min
refit of the original split against section 3: 270,000 of 270,000 rows matched, largest NLL difference 0.0e+00: identical
saved: per_image_nll_G1-split1.csv.gz, per_image_nll_G1-split2.csv.gz, per_image_nll_G1-split3.csv.gz, per_image_nll_G1-split4.csv.gz, per_image_nll_G1-split5.csv.gz, per_image_nll_G1-split6.csv.gz, per_image_nll_G1-split7.csv.gz, per_image_nll_G1-split8.csv.gz, per_image_nll_G1-split9.csv.gz, per_image_nll_G1.csv.gz


In [13]:
#@title 6.5 Analysis library: imports { display-mode: "form" }
import shutil, tempfile
from types import SimpleNamespace
from scipy import stats
from IPython.display import display, Markdown
print("analysis library loading")

analysis library loading


In [14]:
#@title 6.5a Analysis library: data { display-mode: "form" }
REQUIRED = ["dataset", "backbone", "run", "fold", "image_id", "family", "severity", "arm", "nll"]
TREATMENTS = ["C4", "C6c", "C7", "C7q", "C8", "C8b"]   # the quality-conditioned arms of Table 5
BASELINES = ["C2", "C5"]
DEFAULT_ALPHA, DEFAULT_SEED, NAN = 0.05, 20261101, float("nan")


def validate_long(df):
    """Check columns and duplicates; shorten long arm names ("C4 linear / raw q" -> "C4")."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"input is missing columns {missing}; see section 0")
    df = df.copy()
    df["arm"] = df["arm"].astype(str).str.split().str[0]
    for c in ("dataset", "backbone", "run", "family"):
        df[c] = df[c].astype(str)
    dup = df.duplicated(["dataset", "backbone", "run", "image_id", "family", "severity", "arm"])
    if dup.any():
        raise ValueError(f"{int(dup.sum())} duplicated (run, image, family, severity, arm) rows")
    return df


def load_long(paths):
    """Read one or more long per-image tables (CSV, CSV.GZ or Parquet)."""
    frames = [pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p) for p in paths]
    return validate_long(pd.concat(frames, ignore_index=True))


def inventory(df):
    """One row per dataset / backbone / run, listing arms whose image x family x severity design is incomplete."""
    rows = []
    for key, g in df.groupby(["dataset", "backbone", "run"], sort=False):
        test = g[g["family"] != "clean"]
        n_img, n_fam, n_sev = test["image_id"].nunique(), test["family"].nunique(), test["severity"].nunique()
        per_arm = test.groupby("arm").size()
        full = {a for a, n in per_arm.items() if n == n_img * n_fam * n_sev}
        rows.append(dict(dataset=key[0], backbone=key[1], run=key[2], images=n_img, families=n_fam,
                         severities=n_sev, folds=test["fold"].nunique(), arms=" ".join(sorted(per_arm.index)),
                         incomplete_arms=" ".join(sorted(set(per_arm.index) - full)) or "none",
                         has_correct="correct" in g.columns, has_clean_rows=bool((g["family"] == "clean").any())))
    return pd.DataFrame(rows)


def paired_cube(g, A, B):
    """Complete image x family x severity cube of nll[A] - nll[B] for one dataset / backbone / run."""
    arms = set(g["arm"].unique())
    if A not in arms or B not in arms:
        return None
    test = g[g["family"] != "clean"]
    fams = list(pd.unique(test["family"]))
    sevs = np.sort(test["severity"].unique())
    imgs = np.sort(test["image_id"].unique())
    a = test[test["arm"] == A].set_index(["image_id", "family", "severity"])["nll"]
    b = test[test["arm"] == B].set_index(["image_id", "family", "severity"])["nll"]
    full = pd.MultiIndex.from_product([imgs, fams, sevs], names=["image_id", "family", "severity"])
    d = (a - b).reindex(full)
    if d.isna().any():
        raise ValueError(f"{A} vs {B}: {int(d.isna().sum())} of {len(d)} image x family x severity "
                         "cells are missing; the crossed analysis needs the complete design")
    D3 = d.to_numpy().reshape(len(imgs), len(fams), len(sevs))

    if (test.groupby("image_id")["fold"].nunique() > 1).any():
        raise ValueError("an image appears in more than one test fold")
    folds = test.groupby("image_id")["fold"].first().reindex(imgs).to_numpy()

    out = dict(D3=D3, families=fams, severities=sevs, images=imgs, folds=folds,
               D_clean=None, acc=None, acc_clean=None, warnings=[])
    clean = g[g["family"] == "clean"]
    if len(clean):
        ca = clean[clean["arm"] == A].set_index("image_id")["nll"].reindex(imgs)
        cb = clean[clean["arm"] == B].set_index("image_id")["nll"].reindex(imgs)
        if not (ca.isna().any() or cb.isna().any()):
            out["D_clean"] = (ca - cb).to_numpy()
    if "correct" in g.columns:
        sub = g[g["arm"].isin([A, B])]
        varies = sub.groupby(["image_id", "family", "severity"])["correct"].nunique()
        if (varies > 1).any():
            out["warnings"].append(f"'correct' differs between {A} and {B} in {int((varies > 1).sum())} "
                                   f"cells, so some arm changes predictions; accuracy is taken from {B}")
        cb_ = test[test["arm"] == B].set_index(["image_id", "family", "severity"])["correct"].reindex(full)
        if not cb_.isna().any():
            out["acc"] = cb_.to_numpy().reshape(len(imgs), len(fams), len(sevs)).mean(0)
        if len(clean):
            cc = clean[clean["arm"] == B].set_index("image_id")["correct"].reindex(imgs)
            if not cc.isna().any():
                out["acc_clean"] = float(cc.mean())
    return out

In [15]:
#@title 6.5b Analysis library: crossed model { display-mode: "form" }
def mean_squares(D):
    """Two-way ANOVA mean squares of a complete m x F matrix (images x families)."""
    D = np.asarray(D, float)
    m, F = D.shape
    gm = D.mean(); rm = D.mean(1); cm = D.mean(0)
    MSA = m * ((cm - gm) ** 2).sum() / (F - 1)
    MSB = F * ((rm - gm) ** 2).sum() / (m - 1)
    res = D - rm[:, None] - cm[None, :] + gm
    MSE = (res ** 2).sum() / ((m - 1) * (F - 1))
    return m, F, gm, MSA, MSB, MSE


def _satterthwaite(terms):
    """terms: list of (coefficient, mean square, df). Returns (value, df)."""
    v = sum(c * ms for c, ms, _ in terms)
    den = sum((c * ms) ** 2 / nu for c, ms, nu in terms)
    return v, (v ** 2 / den if den > 0 else float("inf"))


def two_way(D, alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED):
    """Crossed two-way random-effects analysis of a complete m x F matrix.

    E[MS_A] = s_e^2 + m s_a^2, E[MS_B] = s_e^2 + F s_b^2, E[MS_E] = s_e^2.
    SE_family^2 = MS_A/(mF), SE_image^2 = MS_B/(mF), SE ratio R = sqrt(MS_A/MS_B), and
    R^2 (s_e^2 + F s_b^2)/(s_e^2 + m s_a^2) ~ F(F-1, m-1): an exact CI for the true ratio.
    The generalized CI for rho_family follows Burdick, Borror and Montgomery (2005)."""
    m, F, gm, MSA, MSB, MSE = mean_squares(D)
    nA, nB, nE = F - 1, m - 1, (m - 1) * (F - 1)
    s2a = max((MSA - MSE) / m, 0.0); s2b = max((MSB - MSE) / F, 0.0); s2e = MSE
    tot = s2a + s2b + s2e
    se_fam = math.sqrt(MSA / (m * F)); se_img = math.sqrt(MSB / (m * F))
    R = se_fam / se_img if se_img > 0 else float("inf")
    fq = lambda p, a, b: stats.f.ppf(p, a, b)
    r_lo = R / math.sqrt(fq(1 - alpha / 2, nA, nB)); r_hi = R / math.sqrt(fq(alpha / 2, nA, nB))
    FA = MSA / MSE
    lam_lo = max((FA / fq(1 - alpha / 2, nA, nE) - 1) / m, 0.0)
    lam_hi = max((FA / fq(alpha / 2, nA, nE) - 1) / m, 0.0)

    # two-component quantities (Table 5 definitions): s_e^2 = mF SE_image^2 = MS_B and
    # s_a^2 = var(family means) - s_e^2/m = (MS_A - MS_B)/m; the SE ratio maps one to one to rho
    s2a_2c = max((MSA - MSB) / m, 0.0); s2e_2c = MSB
    rho_2c = s2a_2c / (s2a_2c + s2e_2c) if s2a_2c + s2e_2c > 0 else 0.0
    g = lambda th: max(((th - 1) / m) / (1 + (th - 1) / m), 0.0)

    # variance of the grand mean over all three sources (truncated moment estimates)
    terms = []
    if MSA > MSE: terms.append((1.0, MSA, nA))
    if MSB > MSE: terms.append((1.0, MSB, nB))
    k = 1 - int(MSA > MSE) - int(MSB > MSE)
    if k: terms.append((float(k), MSE, nE))
    v_full, df_full = _satterthwaite(terms)
    se_full = math.sqrt(max(v_full, 0.0) / (m * F))
    t_full = stats.t.ppf(1 - alpha / 2, df_full) if np.isfinite(df_full) else stats.norm.ppf(1 - alpha / 2)
    t_fam = stats.t.ppf(1 - alpha / 2, nA); z = stats.norm.ppf(1 - alpha / 2)

    out = dict(m=m, F=F, mean=gm, MS_A=MSA, MS_B=MSB, MS_E=MSE,
               sigma_a=math.sqrt(s2a), sigma_b=math.sqrt(s2b), sigma_e=math.sqrt(s2e),
               rho_family=s2a / tot if tot > 0 else 0.0, rho_image=s2b / tot if tot > 0 else 0.0,
               F_family=FA, p_F_family=stats.f.sf(FA, nA, nE),
               var_ratio_a_e_lo=lam_lo, var_ratio_a_e_hi=lam_hi,
               se_family=se_fam, se_image=se_img, se_ratio=R, se_ratio_lo=r_lo, se_ratio_hi=r_hi,
               p_se_ratio=stats.f.sf(R ** 2, nA, nB),
               sigma_a_2c=math.sqrt(s2a_2c), sigma_e_2c=math.sqrt(s2e_2c), rho_2c=rho_2c,
               rho_2c_lo=g(r_lo ** 2), rho_2c_hi=g(r_hi ** 2),
               se_full=se_full, df_full=df_full, full_over_family=se_full / se_fam if se_fam > 0 else NAN,
               ci_family=(gm - t_fam * se_fam, gm + t_fam * se_fam),
               ci_image=(gm - z * se_img, gm + z * se_img),
               ci_full=(gm - t_full * se_full, gm + t_full * se_full))
    if n_gpq:
        rng = np.random.default_rng(seed)
        GE = nE * MSE / rng.chisquare(nE, n_gpq)
        GA = nA * MSA / rng.chisquare(nA, n_gpq)
        GB = nB * MSB / rng.chisquare(nB, n_gpq)
        ga = np.maximum((GA - GE) / m, 0.0); gb = np.maximum((GB - GE) / F, 0.0)
        rho = ga / (ga + gb + GE)
        out["rho_family_lo"], out["rho_family_hi"] = np.quantile(rho, [alpha / 2, 1 - alpha / 2])
    return out


def table5_row(tw):
    """The four quantities printed in Table 5, from a two_way() result."""
    return dict(sigma_a=tw["sigma_a_2c"], sigma_e=tw["sigma_e_2c"], rho=tw["rho_2c"],
                se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"])


def image_bootstrap(D, B=2000, alpha=DEFAULT_ALPHA, seed=DEFAULT_SEED):
    """Percentile bootstrap over images, as in the manuscript (families held fixed)."""
    r = np.asarray(D, float).mean(1); m = len(r)
    rng = np.random.default_rng(seed)
    means = np.empty(B)
    for s in range(0, B, 500):
        b = min(500, B - s)
        means[s:s + b] = r[rng.integers(0, m, (b, m))].mean(1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return dict(se_boot=float(means.std(ddof=1)), ci_boot=(float(lo), float(hi)),
                se_from_width=float((hi - lo) / (2 * stats.norm.ppf(1 - alpha / 2))))

In [16]:
#@title 6.5c Analysis library: tests of the family effect { display-mode: "form" }
def _permuted(D, rng, b):
    m, F = D.shape
    idx = np.argsort(rng.random((b, m, F)), axis=2)
    return np.take_along_axis(np.broadcast_to(D, (b, m, F)), idx, axis=2)


def perm_test(D, n_perm=20000, seed=DEFAULT_SEED, batch=500):
    """Section 6.5 test: variance of the family means, family labels permuted within images.

    Exact under H0: within each image, the joint law of (d_i1, ..., d_iF) is invariant under
    permutation of the family index, images independent. Family-specific variances violate
    this, which is why the two tests below are also reported."""
    D = np.asarray(D, float)
    rng = np.random.default_rng(seed)
    obs = float(D.mean(0).var(ddof=1))
    null = np.empty(n_perm)
    for s in range(0, n_perm, batch):
        b = min(batch, n_perm - s)
        null[s:s + b] = _permuted(D, rng, b).mean(1).var(axis=1, ddof=1)
    med = float(np.median(null))
    return dict(stat=obs, null_median=med, ratio_to_median=obs / med if med > 0 else float("inf"),
                p=float((1 + np.sum(null >= obs)) / (n_perm + 1)), n_perm=n_perm)


def _contrast(F):
    return np.hstack([np.eye(F - 1), -np.ones((F - 1, 1))])


def _t2(P):
    """Hotelling T^2 for H0: equal family means, for a stack P of shape (b, m, F)."""
    b, m, F = P.shape
    Y = P @ _contrast(F).T
    yb = Y.mean(1)
    Yc = Y - yb[:, None, :]
    S = np.einsum("bmi,bmj->bij", Yc, Yc) / (m - 1)
    try:
        sol = np.linalg.solve(S, yb[..., None])[..., 0]
    except np.linalg.LinAlgError:
        sol = np.einsum("bij,bj->bi", np.linalg.pinv(S), yb)
    return m * np.einsum("bi,bi->b", yb, sol)


def hotelling_rm(D):
    """Repeated-measures Hotelling T^2: equal family means under any within-image covariance
    (no sphericity, no equal variances). Exact under multivariate normality of the image
    vectors; the chi-square reference is its large-m (Wald-type statistic) limit."""
    D = np.asarray(D, float)
    m, F = D.shape
    T2 = float(_t2(D[None])[0])
    Fst = (m - F + 1) / ((m - 1) * (F - 1)) * T2
    return dict(T2=T2, F=Fst, df1=F - 1, df2=m - F + 1, p_F=float(stats.f.sf(Fst, F - 1, m - F + 1)),
                p_chi2=float(stats.chi2.sf(T2, F - 1)))


def wts_perm_test(D, n_perm=5000, seed=DEFAULT_SEED, batch=250):
    """Studentized permutation test (Friedrich, Brunner and Pauly 2017): the Wald-type statistic
    recomputed, covariance included, on every within-image permutation. Exact when the
    within-image exchangeability of perm_test holds, and asymptotically valid for equal family
    means under any covariance when it does not (Chung and Romano 2013)."""
    D = np.asarray(D, float)
    rng = np.random.default_rng(seed + 1)
    obs = float(_t2(D[None])[0])
    null = np.empty(n_perm)
    for s in range(0, n_perm, batch):
        b = min(batch, n_perm - s)
        null[s:s + b] = _t2(_permuted(D, rng, b))
    return dict(T2=obs, p=float((1 + np.sum(null >= obs)) / (n_perm + 1)), n_perm=n_perm)

In [17]:
#@title 6.5d Analysis library: sensitivity analyses { display-mode: "form" }
def lofo(D, families, alpha=DEFAULT_ALPHA):
    """Leave one family out: family-level t test, image-level z test, SE ratio and F test."""
    D = np.asarray(D, float)
    rows = []
    for j in [None] + list(range(D.shape[1])):
        Dj = D if j is None else np.delete(D, j, axis=1)
        tw = two_way(Dj, alpha=alpha, n_gpq=0)
        t = stats.ttest_1samp(Dj.mean(0), 0.0)
        zi = tw["mean"] / tw["se_image"]
        rows.append(dict(omitted="none" if j is None else families[j], n_families=Dj.shape[1],
                         mean=tw["mean"], t_family=float(t.statistic), p_family=float(t.pvalue),
                         ci_family_lo=tw["ci_family"][0], ci_family_hi=tw["ci_family"][1],
                         p_image=float(2 * stats.norm.sf(abs(zi))),
                         se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                         F_family=tw["F_family"], p_F_family=tw["p_F_family"]))
    return pd.DataFrame(rows)


_FOLD_KEYS = ("images_per_fold", "icc_fold", "icc_fold_lo", "icc_fold_hi", "F_fold", "p_fold", "var_fold",
              "var_fold_hi", "design_effect", "design_effect_hi", "se_image_iid", "se_image_deff",
              "se_image_deff_hi", "se_image_cluster_robust", "se_image_two_stage")


def fold_clustered(D, folds, alpha=DEFAULT_ALPHA, B=2000, seed=DEFAULT_SEED):
    """Does sharing the probe and calibrators within a fold inflate the image-level SE?

    One-way ANOVA of the image means on fold gives the within-fold intraclass correlation and
    the design effect 1 + (n0 - 1) ICC; a cluster-robust SE and a two-stage bootstrap (folds,
    then images within folds) are cross-checks. With five folds all of them are rough, which
    is why the repeated-partition table is the more informative check."""
    D = np.asarray(D, float)
    r = D.mean(1); m = len(r)
    ks = np.unique(folds); K = len(ks)
    if K < 2:
        return dict(n_folds=K, **{k: NAN for k in _FOLD_KEYS})
    groups = [r[folds == k] for k in ks]
    nk = np.array([len(x) for x in groups])
    gm = r.mean(); mk = np.array([x.mean() for x in groups])
    MSb = (nk * (mk - gm) ** 2).sum() / (K - 1)
    MSw = sum(((x - x.mean()) ** 2).sum() for x in groups) / (m - K)
    n0 = (m - (nk ** 2).sum() / m) / (K - 1)
    s2f = max((MSb - MSw) / n0, 0.0); icc = s2f / (s2f + MSw)
    Fo = MSb / MSw
    FL = Fo / stats.f.ppf(1 - alpha / 2, K - 1, m - K); FU = Fo / stats.f.ppf(alpha / 2, K - 1, m - K)
    icc_lo = max((FL - 1) / (FL + n0 - 1), 0.0); icc_hi = max((FU - 1) / (FU + n0 - 1), 0.0)
    se_iid = r.std(ddof=1) / math.sqrt(m)
    u = np.array([(x - gm).sum() for x in groups])
    se_cr = math.sqrt(K / (K - 1) * (u ** 2).sum()) / m
    rng = np.random.default_rng(seed + 2)
    bm = np.empty(B)
    for b in range(B):
        pick = rng.integers(0, K, K)
        bm[b] = np.concatenate([groups[c][rng.integers(0, nk[c], nk[c])] for c in pick]).mean()
    deff = 1 + (n0 - 1) * icc; deff_hi = 1 + (n0 - 1) * icc_hi
    s2f_hi = icc_hi * MSw / (1 - icc_hi) if icc_hi < 1 else float("inf")
    return dict(n_folds=K, images_per_fold=n0, icc_fold=icc, icc_fold_lo=icc_lo, icc_fold_hi=icc_hi,
                F_fold=Fo, p_fold=float(stats.f.sf(Fo, K - 1, m - K)),
                var_fold=s2f / K, var_fold_hi=s2f_hi / K,       # added to the variance of the grand mean
                design_effect=deff, design_effect_hi=deff_hi,
                se_image_iid=se_iid, se_image_deff=se_iid * math.sqrt(deff),
                se_image_deff_hi=se_iid * math.sqrt(deff_hi),
                se_image_cluster_robust=se_cr, se_image_two_stage=float(bm.std(ddof=1)))


def split_stem(run, tag):
    """'G1-split3' -> 'G1'; a run without the tag is its own stem."""
    return run.split(tag)[0] if tag and tag in run else run


def split_variability(rows, tag="-split"):
    """How far does each mean move when only the fold partition changes?

    Partitions of one grid are the runs '<grid><tag><k>' plus the base run '<grid>' (counted as
    partition 0 unless '<grid><tag>0' also exists, which would be the same split twice). The SD
    over partitions measures training-sample variability that the image-level interval ignores;
    it is a lower bound, because the partitions share most training images (Bates, Hastie and
    Tibshirani 2024)."""
    df = pd.DataFrame(rows)
    if df.empty or not tag:
        return None
    df["grid"] = [split_stem(r, tag) for r in df["run"]]
    tagged = df["run"].str.contains(tag, regex=False)
    stems = set(zip(df.loc[tagged, "dataset"], df.loc[tagged, "backbone"], df.loc[tagged, "grid"]))
    if not stems:
        return None
    runs = set(zip(df["dataset"], df["backbone"], df["run"]))
    keep = []
    for i, r in df.iterrows():
        if (r["dataset"], r["backbone"], r["grid"]) not in stems:
            continue
        if tag in r["run"] or (r["dataset"], r["backbone"], f"{r['grid']}{tag}0") not in runs:
            keep.append(i)
    out = []
    for (ds, bb, gr, comp), g in df.loc[keep].groupby(["dataset", "backbone", "grid", "comparison"], sort=False):
        if len(g) < 2:
            continue
        sd = float(g["mean"].std(ddof=1))
        se_i, se_f = float(g["se_image"].mean()), float(g["se_family"].mean())
        out.append(dict(dataset=ds, backbone=bb, grid=gr, comparison=comp, n_splits=len(g),
                        mean_of_means=float(g["mean"].mean()), sd_across_splits=sd, se_image=se_i,
                        se_family=se_f, sd_over_se_image=sd / se_i, sd_over_se_family=sd / se_f,
                        se_ratio_min=float(g["se_ratio"].min()), se_ratio_max=float(g["se_ratio"].max())))
    return pd.DataFrame(out) if out else None


def per_severity(D3, severities, alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED):
    """Crossed components separately at each severity level, plus the severity average."""
    D3 = np.asarray(D3, float)
    rows = []
    labels = [str(s) for s in severities] + ["mean"]
    for j, s in enumerate(labels):
        tw = two_way(D3[:, :, j] if s != "mean" else D3.mean(2), alpha=alpha, n_gpq=n_gpq, seed=seed)
        rows.append(dict(severity=s, mean=tw["mean"], sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"],
                         sigma_e=tw["sigma_e"], rho_family=tw["rho_family"],
                         rho_family_lo=tw.get("rho_family_lo"), rho_family_hi=tw.get("rho_family_hi"),
                         se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                         F_family=tw["F_family"], p_F_family=tw["p_F_family"]))
    return pd.DataFrame(rows)


def _locate(loss, tau):
    """First crossing of tau on a severity ladder of losses -> (j0, j1, w) or None."""
    for j in range(1, len(loss)):
        lo, hi = loss[j - 1], loss[j]
        if lo <= tau <= hi and hi > lo:
            return j - 1, j, (tau - lo) / (hi - lo)
    return None


def amplitude_matched(D3, acc, families, severities, targets=None, D_clean=None, acc_clean=None,
                      method="interp", alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED, min_families=3):
    """Re-estimate the family component with families compared at matched damage.

    Damage is the probe's accuracy loss, which no arm changes (all arms rescale logits by a
    positive temperature), so the matching variable is not the analysed NLL difference.
    method="interp": for each family, interpolate the per-image differences linearly between
    the two severity levels whose losses bracket the target (clean = zero loss when clean rows
    exist); families whose ladder never reaches the target are left out at that target.
    method="nearest": use each family's level closest to the target and report the mismatch."""
    D3 = np.asarray(D3, float); acc = np.asarray(acc, float)
    m, F, S = D3.shape
    base = acc_clean if acc_clean is not None else float(np.max(acc))
    loss = base - acc                                   # F x S, larger = more damage
    levels = list(severities)
    if D_clean is not None and acc_clean is not None:
        loss = np.hstack([np.zeros((F, 1)), loss])
        D3 = np.concatenate([np.asarray(D_clean, float)[:, None, None].repeat(F, 1), D3], axis=2)
        levels = [0] + levels
    if targets is None:
        lo = max(loss[f].min() for f in range(F)); hi = min(loss[f].max() for f in range(F))
        targets = list(np.linspace(lo, hi, 7)[1:-1]) if hi > lo else list(np.quantile(loss, [0.2, 0.4, 0.6, 0.8]))
    rows = []
    for tau in targets:
        cols, used, where, mism = [], [], [], []
        for f in range(F):
            if method == "nearest":
                j = int(np.argmin(np.abs(loss[f] - tau)))
                cols.append(D3[:, f, j]); used.append(families[f]); where.append(str(levels[j]))
                mism.append(float(loss[f, j] - tau))
                continue
            hit = _locate(loss[f], tau)
            if hit is None:
                continue
            j0, j1, w = hit
            cols.append((1 - w) * D3[:, f, j0] + w * D3[:, f, j1]); used.append(families[f])
            where.append(f"{levels[j0]}+{w:.2f}")
        row = dict(target_loss=float(tau), n_families=len(cols), families=";".join(used),
                   position=";".join(where))
        if method == "nearest":
            row["max_abs_mismatch"] = float(np.max(np.abs(mism)))
        if len(cols) >= min_families:
            tw = two_way(np.column_stack(cols), alpha=alpha, n_gpq=n_gpq, seed=seed)
            row.update(mean=tw["mean"], sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"], sigma_e=tw["sigma_e"],
                       rho_family=tw["rho_family"], rho_family_lo=tw.get("rho_family_lo"),
                       rho_family_hi=tw.get("rho_family_hi"), se_ratio=tw["se_ratio"],
                       se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                       F_family=tw["F_family"], p_F_family=tw["p_F_family"])
        rows.append(row)
    return pd.DataFrame(rows), loss

In [18]:
#@title 6.5e Analysis library: driver, report and simulated data { display-mode: "form" }
def default_pairs(arms):
    """The twelve comparisons of Table 5 that the data contain, plus each quality arm against C5o."""
    pairs = [(a, b) for b in BASELINES for a in TREATMENTS if a in arms and b in arms]
    oracle = sorted(x for x in arms if x.startswith("C5") and x.endswith("o"))
    pairs += [(a, o) for o in oracle for a in TREATMENTS if a in arms]
    return pairs


def primary_runs(keys, tag):
    """Per dataset / backbone / grid, the run that gets the full analyses: the base run ('G1')
    if present, else '<grid><tag>0', else the first repeated partition."""
    by = {}
    for ds, bb, run in keys:
        by.setdefault((ds, bb, split_stem(run, tag)), []).append(run)
    prim = {}
    for (ds, bb, stem), runs in by.items():
        zero = f"{stem}{tag}0"
        prim[(ds, bb, stem)] = stem if stem in runs else (zero if zero in runs else sorted(runs)[0])
    return prim


def _div(a, b):
    return a / b if b and np.isfinite(b) else NAN


def analyse_pair(cube, A, B, cfg, light=False):
    """Every analysis for one comparison. Light mode (repeated partitions) keeps only the mean,
    the standard errors and the fold check."""
    D3 = cube["D3"]; D = D3.mean(2); fams = cube["families"]
    tw = two_way(D, alpha=cfg.alpha, n_gpq=0 if light else cfg.n_gpq, seed=cfg.seed)
    fc = fold_clustered(D, cube["folds"], alpha=cfg.alpha, B=200 if light else cfg.n_boot, seed=cfg.seed)
    bt = dict(se_from_width=NAN, ci_boot=(NAN, NAN)); pm = dict(p=NAN, ratio_to_median=NAN)
    t2 = dict(T2=NAN, p_F=NAN); wt = dict(p=NAN); lf = ps = am = None
    if not light:
        bt = image_bootstrap(D, B=cfg.n_boot, alpha=cfg.alpha, seed=cfg.seed)
        pm = perm_test(D, n_perm=cfg.n_perm, seed=cfg.seed)
        t2 = hotelling_rm(D)
        wt = wts_perm_test(D, n_perm=cfg.n_perm_wts, seed=cfg.seed)
        lf = lofo(D, fams, alpha=cfg.alpha)
        ps = per_severity(D3, cube["severities"], alpha=cfg.alpha, n_gpq=cfg.n_gpq, seed=cfg.seed)
        if cube["acc"] is not None:
            am, _ = amplitude_matched(D3, cube["acc"], fams, cube["severities"], targets=cfg.targets,
                                      D_clean=cube["D_clean"], acc_clean=cube["acc_clean"],
                                      method=cfg.amp_method, alpha=cfg.alpha, n_gpq=cfg.n_gpq, seed=cfg.seed)
    lofo_rows = lf.iloc[1:] if lf is not None else None
    row = dict(comparison=f"{A} vs {B}", light=light, m=tw["m"], F=tw["F"], S=D3.shape[2], mean=tw["mean"],
               se_family=tw["se_family"], se_image=tw["se_image"], se_image_boot=bt["se_from_width"],
               se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
               p_se_ratio=tw["p_se_ratio"],
               t5_sigma_a=tw["sigma_a_2c"], t5_sigma_e=tw["sigma_e_2c"], t5_rho=tw["rho_2c"],
               t5_rho_lo=tw["rho_2c_lo"], t5_rho_hi=tw["rho_2c_hi"],
               sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"], sigma_e=tw["sigma_e"],
               rho_family=tw["rho_family"], rho_family_lo=tw.get("rho_family_lo", NAN),
               rho_family_hi=tw.get("rho_family_hi", NAN), F_family=tw["F_family"], p_F_family=tw["p_F_family"],
               se_full=tw["se_full"], df_full=tw["df_full"], full_over_family=tw["full_over_family"],
               ci_family_lo=tw["ci_family"][0], ci_family_hi=tw["ci_family"][1],
               ci_full_lo=tw["ci_full"][0], ci_full_hi=tw["ci_full"][1],
               ci_image_boot_lo=bt["ci_boot"][0], ci_image_boot_hi=bt["ci_boot"][1],
               perm_p=pm["p"], perm_ratio_to_median=pm["ratio_to_median"],
               T2=t2["T2"], T2_p_F=t2["p_F"], wts_perm_p=wt["p"],
               icc_fold=fc["icc_fold"], icc_fold_hi=fc["icc_fold_hi"], design_effect=fc["design_effect"],
               design_effect_hi=fc["design_effect_hi"],
               se_ratio_fold_adjusted=_div(tw["se_family"], fc["se_image_deff"]),
               se_ratio_fold_adjusted_conservative=_div(tw["se_family"], fc["se_image_deff_hi"]),
               se_image_cluster_robust=fc["se_image_cluster_robust"],
               se_image_two_stage=fc["se_image_two_stage"],
               fold_share_of_family_var=_div(fc["var_fold"], tw["se_family"] ** 2),
               fold_share_of_family_var_hi=_div(fc["var_fold_hi"], tw["se_family"] ** 2),
               se_full_with_fold=math.sqrt(tw["se_full"] ** 2 + fc["var_fold"]) if np.isfinite(fc["var_fold"]) else NAN,
               se_full_with_fold_hi=math.sqrt(tw["se_full"] ** 2 + fc["var_fold_hi"]) if np.isfinite(fc["var_fold_hi"]) else NAN,
               lofo_min_p_family=float(lofo_rows["p_family"].min()) if lofo_rows is not None else NAN,
               lofo_family_at_min=str(lofo_rows.loc[lofo_rows["p_family"].idxmin(), "omitted"]) if lofo_rows is not None else "",
               lofo_min_se_ratio=float(lofo_rows["se_ratio"].min()) if lofo_rows is not None else NAN)
    tabs = dict(lofo=lf, per_severity=ps, amplitude_matched=am,
                family_by_severity_means=pd.DataFrame(D3.mean(0), index=fams,
                                                      columns=[str(s) for s in cube["severities"]]))
    return row, tabs


def _f(x, d=3):
    try:
        return f"{x:.{d}f}" if x is not None and np.isfinite(x) else "n/a"
    except TypeError:
        return "n/a"


def _p(x):
    if x is None or not np.isfinite(x):
        return "n/a"
    return f"{x:.1e}" if x < 1e-3 else f"{x:.3f}"


def write_report(groups, cfg, split=None):
    """Markdown report: one section per dataset / backbone / run, then the repeated partitions."""
    L = ["# Revision analyses", ""]
    if getattr(cfg, "demo", False):
        L += ["**Demo on simulated data. These numbers are not results.**", ""]
    L += [f"alpha {cfg.alpha}; {cfg.n_perm} permutations (plain), {cfg.n_perm_wts} (studentized); "
          f"{cfg.n_gpq} generalized-pivot draws; {cfg.n_boot} bootstrap replicates; seed {cfg.seed}.", ""]
    for key, rows, warns in groups:
        if not rows:
            if warns:
                L += [f"## {' / '.join(key)}", ""] + [f"Warning: {w}" for w in warns] + [""]
            continue
        L += [f"## {' / '.join(key)}", ""]
        if all(r["light"] for r in rows):
            L += ["Repeated fold partition, light mode: only the mean, the standard errors and the fold check "
                  "were computed. See the repeated-partition table at the end.", ""]
            L += [f"Warning: {w}" for w in warns] + ([""] if warns else [])
            continue
        L += [f"m = {rows[0]['m']} images, F = {rows[0]['F']} families, {rows[0]['S']} severity levels.", ""]
        L += [f"Warning: {w}" for w in warns] + ([""] if warns else [])
        L += ["### Table 5 quantities, recomputed (two-component, Section 5.3 definitions)", "",
              "| comparison | sigma_a | sigma_e | rho [95% exact] | SE ratio [95% exact] | SE ratio, bootstrap SE |",
              "|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['t5_sigma_a'])} | {_f(r['t5_sigma_e'])} | {_f(r['t5_rho'])} "
                     f"[{_f(r['t5_rho_lo'])}, {_f(r['t5_rho_hi'])}] | {_f(r['se_ratio'], 2)} "
                     f"[{_f(r['se_ratio_lo'], 2)}, {_f(r['se_ratio_hi'], 2)}] | "
                     f"{_f(_div(r['se_family'], r['se_image_boot']), 2)} |")
        L += ["", "### Crossed decomposition and tests of the family effect", "",
              "| comparison | sigma_a | sigma_b | sigma_e | rho_family [95% GCI] | F test p | permutation p "
              "(obs/median) | Hotelling p | studentized perm p | full SE / family SE |",
              "|---|---|---|---|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['sigma_a'])} | {_f(r['sigma_b'])} | {_f(r['sigma_e'])} | "
                     f"{_f(r['rho_family'])} [{_f(r['rho_family_lo'])}, {_f(r['rho_family_hi'])}] | "
                     f"{_p(r['p_F_family'])} | {_p(r['perm_p'])} ({_f(r['perm_ratio_to_median'], 1)}) | "
                     f"{_p(r['T2_p_F'])} | {_p(r['wts_perm_p'])} | {_f(r['full_over_family'], 4)} |")
        L += ["", "### Intervals for the mean difference", "",
              "| comparison | mean | image bootstrap 95% | family t 95% | full crossed 95% (df) |",
              "|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['mean'], 4)} | [{_f(r['ci_image_boot_lo'], 4)}, "
                     f"{_f(r['ci_image_boot_hi'], 4)}] | [{_f(r['ci_family_lo'], 4)}, {_f(r['ci_family_hi'], 4)}] "
                     f"| [{_f(r['ci_full_lo'], 4)}, {_f(r['ci_full_hi'], 4)}] ({_f(r['df_full'], 1)}) |")
        L += ["", "### Leave one family out and fold dependence", "",
              "| comparison | smallest LOFO family p (family omitted) | smallest LOFO SE ratio | fold ICC [upper] "
              "| design effect [upper] | SE ratio, fold-adjusted [conservative] |",
              "|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_p(r['lofo_min_p_family'])} ({r['lofo_family_at_min']}) | "
                     f"{_f(r['lofo_min_se_ratio'], 2)} | {_f(r['icc_fold'], 4)} [{_f(r['icc_fold_hi'], 4)}] | "
                     f"{_f(r['design_effect'], 2)} [{_f(r['design_effect_hi'], 2)}] | "
                     f"{_f(r['se_ratio_fold_adjusted'], 2)} [{_f(r['se_ratio_fold_adjusted_conservative'], 2)}] |")
        L += ["", "Fold component as a share of the family-level variance: "
              + "; ".join(f"{r['comparison']} {_f(r['fold_share_of_family_var'], 3)} "
                          f"[upper {_f(r['fold_share_of_family_var_hi'], 3)}]" for r in rows) + ".", "",
              "Per-comparison tables (leave one family out, per severity, amplitude matched, family x severity "
              "means) are in the comparison folders of the output directory.", ""]
    if split is not None and len(split):
        L += ["## Repeated fold partitions", "",
              "| dataset / backbone / grid | comparison | partitions | SD of the mean over partitions | SE image | "
              "SE family | SD / SE image | SD / SE family | SE ratio range |", "|---|---|---|---|---|---|---|---|---|"]
        for _, r in split.iterrows():
            L.append(f"| {r['dataset']} / {r['backbone']} / {r['grid']} | {r['comparison']} | {r['n_splits']} | "
                     f"{_f(r['sd_across_splits'], 4)} | {_f(r['se_image'], 4)} | {_f(r['se_family'], 4)} | "
                     f"{_f(r['sd_over_se_image'], 2)} | {_f(r['sd_over_se_family'], 2)} | "
                     f"{_f(r['se_ratio_min'], 2)} to {_f(r['se_ratio_max'], 2)} |")
        L.append("")
    return "\n".join(L)


def run_all(df, cfg):
    """Run every comparison in every dataset / backbone / run; write CSVs and report.md to cfg.out_dir."""
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    tag = cfg.split_tag

    def order(k):                     # base run first, then its partitions in numeric order
        stem = split_stem(k[2], tag)
        tail = k[2][len(stem) + len(tag):] if tag and tag in k[2] else ""
        return (k[0], k[1], stem, k[2] != stem, int(tail) if tail.isdigit() else -1, k[2])

    keys = sorted(df[["dataset", "backbone", "run"]].drop_duplicates().itertuples(index=False, name=None), key=order)
    prim = primary_runs(keys, tag)
    grouped = df.groupby(["dataset", "backbone", "run"], sort=False)
    jobs = []
    for key in keys:
        g = grouped.get_group(key)
        pairs = cfg.pairs or default_pairs(set(g["arm"].unique()))
        light = (cfg.light_splits and bool(cfg.split_tag) and cfg.split_tag in key[2]
                 and prim[(key[0], key[1], split_stem(key[2], cfg.split_tag))] != key[2])
        jobs += [(key, g, A, B, light) for A, B in pairs]
    groups = {k: [] for k in keys}; warns = {k: set() for k in keys}; rows = []; tables = {}
    for key, g, A, B, light in tqdm(jobs, desc="comparisons"):
        try:
            cube = paired_cube(g, A, B)
        except ValueError as e:
            warns[key].add(str(e)); continue
        if cube is None:
            warns[key].add(f"{A} vs {B}: an arm is missing in this run"); continue
        warns[key].update(cube["warnings"])
        row, tabs = analyse_pair(cube, A, B, cfg, light=light)
        row.update(dataset=key[0], backbone=key[1], run=key[2])
        groups[key].append(row); rows.append(row); tables[(key, row["comparison"])] = tabs
        d = out / "_".join(key).replace("/", "-") / f"{A}_vs_{B}"
        d.mkdir(parents=True, exist_ok=True)
        for name, tab in tabs.items():
            if tab is not None:
                tab.to_csv(d / f"{name}.csv", index=(name == "family_by_severity_means"))
    summary = pd.DataFrame(rows)
    if len(summary):
        cols = ["dataset", "backbone", "run"] + [c for c in summary.columns if c not in ("dataset", "backbone", "run")]
        summary = summary[cols]
        summary.to_csv(out / "summary.csv", index=False)
    split = split_variability(rows, cfg.split_tag) if rows else None
    if split is not None:
        split.to_csv(out / "split_variability.csv", index=False)
    report = write_report([(k, groups[k], sorted(warns[k])) for k in keys], cfg, split)
    (out / "report.md").write_text(report)
    (out / "settings.json").write_text(json.dumps(vars(cfg), indent=2, default=str))
    return dict(summary=summary, tables=tables, split=split, report=report, out=out)


DEMO_FAMILIES = ["motion_blur", "shot_noise", "brightness_drift", "contrast_loss", "vignetting", "vibration_jitter"]


def simulate_long_table(runs=("G1",), arms=("C2", "C4", "C5"), m=900, n_sev=5, seed=0, clean=True,
                        dataset="SIM", backbone="toy"):
    """A simulated per-image NLL table in the export format, for the demo and the self-test.
    Families have unequal severity ladders; each arm adds its own family and image effects."""
    rng = np.random.default_rng(seed)
    F = len(DEMO_FAMILIES); sevs = np.arange(1, n_sev + 1)
    loss = np.linspace(0.02, 0.09, F)[:, None] * sevs[None, :]          # F x S accuracy loss
    b = np.abs(rng.normal(0, 0.2, m))                                    # image difficulty
    fam_eff = {a: (0.0 if a == arms[0] else 1.0) * rng.normal(0, 0.05, F) for a in arms}
    img_eff = {a: rng.normal(0, 0.03, m) for a in arms}
    shift = {a: (0.0 if a == arms[0] else rng.uniform(-0.04, 0.02)) for a in arms}
    correct = (rng.random((m, F, n_sev)) < (0.95 - loss)[None]).astype(int)
    correct_clean = (rng.random(m) < 0.95).astype(int)
    frames = []
    for run in runs:
        folds = rng.permutation(np.arange(m) % 5)
        base = 0.3 + 1.2 * loss[None] + b[:, None, None] + rng.normal(0, 0.15, (m, F, n_sev))
        for a in arms:
            nll = (base + shift[a] + fam_eff[a][None, :, None] + img_eff[a][:, None, None]
                   + rng.normal(0, 0.05, (m, F, n_sev)))
            frames.append(pd.DataFrame(dict(
                dataset=dataset, backbone=backbone, run=run, fold=np.repeat(folds, F * n_sev),
                image_id=np.repeat(np.arange(m), F * n_sev), family=np.tile(np.repeat(DEMO_FAMILIES, n_sev), m),
                severity=np.tile(sevs, m * F), arm=a, nll=nll.reshape(-1), correct=correct.reshape(-1))))
            if clean:
                frames.append(pd.DataFrame(dict(
                    dataset=dataset, backbone=backbone, run=run, fold=folds, image_id=np.arange(m),
                    family="clean", severity=0, arm=a, nll=0.3 + b + shift[a] + img_eff[a] + rng.normal(0, 0.05, m),
                    correct=correct_clean)))
    return validate_long(pd.concat(frames, ignore_index=True))

In [19]:
#@title 6.6 Run the analyses { display-mode: "form" }
if RUN_ANALYSES:
    _s = 10 if QUICK else 1
    CFG = SimpleNamespace(alpha=0.05, seed=20261101, pairs=[], n_perm=max(20000 // _s, 500),
                          n_perm_wts=max(5000 // _s, 300), n_gpq=max(20000 // _s, 2000),
                          n_boot=max(2000 // _s, 300), targets=None, amp_method="interp", split_tag="-split",
                          light_splits=True, out_dir=str(REV / "results"), demo=False)
    DF = validate_long(pd.concat(TABLES.values(), ignore_index=True))
    display(inventory(DF))
    _t0 = time.time()
    RES = run_all(DF, CFG)
    print(f"{len(RES['summary'])} comparisons in {(time.time() - _t0) / 60:.1f} min; results in {CFG.out_dir}")
    display(Markdown(RES["report"]))
else:
    print("RUN_ANALYSES is off; the per-image tables are in", REV)

,dataset,backbone,run,images,families,severities,folds,arms,incomplete_arms,has_correct,has_clean_rows
0,NEU-CLS,resnet50.a1_in1k,G1,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
1,NEU-CLS,resnet50.a1_in1k,G1-split1,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
2,NEU-CLS,resnet50.a1_in1k,G1-split2,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
3,NEU-CLS,resnet50.a1_in1k,G1-split3,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
4,NEU-CLS,resnet50.a1_in1k,G1-split4,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
5,NEU-CLS,resnet50.a1_in1k,G1-split5,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
6,NEU-CLS,resnet50.a1_in1k,G1-split6,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
7,NEU-CLS,resnet50.a1_in1k,G1-split7,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
8,NEU-CLS,resnet50.a1_in1k,G1-split8,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True
9,NEU-CLS,resnet50.a1_in1k,G1-split9,900,6,5,5,C1 C2 C3 C4 C5 C5o C6c C7 C7q C8 C8b,none,True,True


comparisons:   0%|          | 0/180 [00:00<?, ?it/s]

180 comparisons in 2.5 min; results in /content/sdic/phase0/revision/results


# Revision analyses

alpha 0.05; 20000 permutations (plain), 5000 (studentized); 20000 generalized-pivot draws; 2000 bootstrap replicates; seed 20261101.

## NEU-CLS / resnet50.a1_in1k / G1

m = 900 images, F = 6 families, 5 severity levels.

### Table 5 quantities, recomputed (two-component, Section 5.3 definitions)

| comparison | sigma_a | sigma_e | rho [95% exact] | SE ratio [95% exact] | SE ratio, bootstrap SE |
|---|---|---|---|---|---|
| C4 vs C2 | 0.635 | 1.056 | 0.266 [0.122, 0.686] | 18.07 [11.25, 44.35] | 18.69 |
| C6c vs C2 | 0.797 | 1.177 | 0.315 [0.151, 0.735] | 20.35 [12.67, 49.94] | 20.26 |
| C7 vs C2 | 0.438 | 0.620 | 0.333 [0.161, 0.750] | 21.20 [13.20, 52.04] | 21.05 |
| C7q vs C2 | 0.391 | 0.622 | 0.283 [0.132, 0.704] | 18.88 [11.75, 46.32] | 19.03 |
| C8 vs C2 | 0.794 | 2.144 | 0.120 [0.050, 0.454] | 11.15 [6.94, 27.36] | 11.37 |
| C8b vs C2 | 0.440 | 1.138 | 0.130 [0.054, 0.475] | 11.63 [7.24, 28.55] | 11.49 |
| C4 vs C5 | 0.751 | 1.234 | 0.270 [0.125, 0.691] | 18.28 [11.38, 44.87] | 18.74 |
| C6c vs C5 | 0.907 | 1.263 | 0.340 [0.166, 0.757] | 21.57 [13.43, 52.93] | 21.82 |
| C7 vs C5 | 0.567 | 1.015 | 0.238 [0.107, 0.653] | 16.79 [10.45, 41.20] | 16.59 |
| C7q vs C5 | 0.522 | 1.009 | 0.211 [0.094, 0.618] | 15.57 [9.69, 38.21] | 15.64 |
| C8 vs C5 | 0.804 | 1.536 | 0.215 [0.095, 0.623] | 15.73 [9.79, 38.60] | 15.84 |
| C8b vs C5 | 0.448 | 0.746 | 0.265 [0.122, 0.686] | 18.06 [11.24, 44.32] | 18.84 |
| C4 vs C5o | 0.601 | 0.962 | 0.281 [0.131, 0.702] | 18.77 [11.69, 46.07] | 19.35 |
| C6c vs C5o | 0.763 | 1.065 | 0.339 [0.165, 0.756] | 21.50 [13.39, 52.77] | 21.41 |
| C7 vs C5o | 0.404 | 0.549 | 0.351 [0.173, 0.765] | 22.08 [13.75, 54.20] | 22.08 |
| C7q vs C5o | 0.357 | 0.550 | 0.297 [0.140, 0.718] | 19.51 [12.15, 47.88] | 19.25 |
| C8 vs C5o | 0.747 | 1.958 | 0.127 [0.053, 0.469] | 11.49 [7.15, 28.20] | 11.71 |
| C8b vs C5o | 0.391 | 0.974 | 0.139 [0.058, 0.494] | 12.08 [7.52, 29.65] | 12.02 |

### Crossed decomposition and tests of the family effect

| comparison | sigma_a | sigma_b | sigma_e | rho_family [95% GCI] | F test p | permutation p (obs/median) | Hotelling p | studentized perm p | full SE / family SE |
|---|---|---|---|---|---|---|---|---|---|
| C4 vs C2 | 0.635 | 0.000 | 1.134 | 0.239 [0.107, 0.655] | 6.8e-264 | 5.0e-05 (247.5) | 4.7e-74 | 2.0e-04 | 1.0000 |
| C6c vs C2 | 0.797 | 0.000 | 1.262 | 0.285 [0.132, 0.707] | 0.0e+00 | 5.0e-05 (296.0) | 6.7e-84 | 2.0e-04 | 1.0000 |
| C7 vs C2 | 0.438 | 0.000 | 0.710 | 0.275 [0.127, 0.697] | 2.8e-312 | 5.0e-05 (286.0) | 1.3e-93 | 2.0e-04 | 1.0000 |
| C7q vs C2 | 0.391 | 0.000 | 0.691 | 0.242 [0.109, 0.660] | 1.7e-268 | 5.0e-05 (250.9) | 2.7e-76 | 2.0e-04 | 1.0000 |
| C8 vs C2 | 0.795 | 0.601 | 1.559 | 0.185 [0.079, 0.579] | 1.7e-223 | 5.0e-05 (215.6) | 9.9e-70 | 2.0e-04 | 1.0019 |
| C8b vs C2 | 0.440 | 0.257 | 0.948 | 0.167 [0.071, 0.550] | 1.4e-188 | 5.0e-05 (184.8) | 1.9e-68 | 2.0e-04 | 1.0011 |
| C4 vs C5 | 0.751 | 0.000 | 1.501 | 0.200 [0.087, 0.603] | 1.1e-215 | 5.0e-05 (207.7) | 1.8e-59 | 2.0e-04 | 1.0000 |
| C6c vs C5 | 0.906 | 0.000 | 1.620 | 0.238 [0.107, 0.655] | 1.9e-263 | 5.0e-05 (247.1) | 8.1e-74 | 2.0e-04 | 1.0000 |
| C7 vs C5 | 0.566 | 0.000 | 1.180 | 0.187 [0.081, 0.583] | 3.4e-200 | 5.0e-05 (194.9) | 2.3e-71 | 2.0e-04 | 1.0000 |
| C7q vs C5 | 0.522 | 0.000 | 1.150 | 0.171 [0.073, 0.556] | 6.4e-181 | 5.0e-05 (177.5) | 2.9e-58 | 2.0e-04 | 1.0000 |
| C8 vs C5 | 0.804 | 0.306 | 1.341 | 0.255 [0.115, 0.675] | 8.6e-298 | 5.0e-05 (274.8) | 1.6e-64 | 2.0e-04 | 1.0005 |
| C8b vs C5 | 0.448 | 0.000 | 0.872 | 0.209 [0.091, 0.616] | 1.1e-226 | 5.0e-05 (216.2) | 4.0e-56 | 2.0e-04 | 1.0000 |
| C4 vs C5o | 0.601 | 0.000 | 1.113 | 0.226 [0.100, 0.639] | 2.1e-247 | 5.0e-05 (234.8) | 1.2e-68 | 2.0e-04 | 1.0000 |
| C6c vs C5o | 0.763 | 0.000 | 1.237 | 0.275 [0.127, 0.697] | 5.4e-312 | 5.0e-05 (285.7) | 1.5e-81 | 2.0e-04 | 1.0000 |
| C7 vs C5o | 0.404 | 0.000 | 0.697 | 0.251 [0.114, 0.671] | 2.7e-280 | 5.0e-05 (261.1) | 6.2e-88 | 2.0e-04 | 1.0000 |
| C7q vs C5o | 0.357 | 0.000 | 0.673 | 0.220 [0.097, 0.631] | 6.3e-240 | 5.0e-05 (227.0) | 7.8e-71 | 2.0e-04 | 1.0000 |
| C8 vs C5o | 0.748 | 0.527 | 1.473 | 0.186 [0.080, 0.582] | 5.0e-222 | 5.0e-05 (213.7) | 2.1e-67 | 2.0e-04 | 1.0016 |
| C8b vs C5o | 0.391 | 0.184 | 0.863 | 0.164 [0.069, 0.544] | 2.4e-180 | 5.0e-05 (176.1) | 3.3e-66 | 2.0e-04 | 1.0007 |

### Intervals for the mean difference

| comparison | mean | image bootstrap 95% | family t 95% | full crossed 95% (df) |
|---|---|---|---|---|
| C4 vs C2 | 0.1973 | [0.1700, 0.2244] | [-0.4703, 0.8649] | [-0.4703, 0.8649] (5.0) |
| C6c vs C2 | 0.2671 | [0.2356, 0.2987] | [-0.5707, 1.1048] | [-0.5707, 1.1048] (5.0) |
| C7 vs C2 | 0.1136 | [0.0967, 0.1300] | [-0.3463, 0.5735] | [-0.3463, 0.5735] (5.0) |
| C7q vs C2 | 0.0955 | [0.0794, 0.1123] | [-0.3150, 0.5061] | [-0.3150, 0.5061] (5.0) |
| C8 vs C2 | 0.4447 | [0.3887, 0.5008] | [-0.3916, 1.2809] | [-0.3913, 1.2806] (5.0) |
| C8b vs C2 | 0.2181 | [0.1874, 0.2489] | [-0.2449, 0.6810] | [-0.2448, 0.6809] (5.0) |
| C4 vs C5 | 0.0616 | [0.0301, 0.0944] | [-0.7278, 0.8511] | [-0.7278, 0.8511] (5.0) |
| C6c vs C5 | 0.1314 | [0.0979, 0.1645] | [-0.8213, 1.0841] | [-0.8213, 1.0841] (5.0) |
| C7 vs C5 | -0.0221 | [-0.0516, 0.0032] | [-0.6179, 0.5737] | [-0.6179, 0.5737] (5.0) |
| C7q vs C5 | -0.0402 | [-0.0675, -0.0139] | [-0.5894, 0.5091] | [-0.5894, 0.5091] (5.0) |
| C8 vs C5 | 0.3090 | [0.2688, 0.3502] | [-0.5364, 1.1543] | [-0.5363, 1.1543] (5.0) |
| C8b vs C5 | 0.0824 | [0.0630, 0.1012] | [-0.3887, 0.5534] | [-0.3887, 0.5534] (5.0) |
| C4 vs C5o | 0.1501 | [0.1253, 0.1751] | [-0.4817, 0.7820] | [-0.4817, 0.7820] (5.0) |
| C6c vs C5o | 0.2199 | [0.1913, 0.2484] | [-0.5816, 1.0214] | [-0.5816, 1.0214] (5.0) |
| C7 vs C5o | 0.0664 | [0.0514, 0.0807] | [-0.3581, 0.4909] | [-0.3581, 0.4909] (5.0) |
| C7q vs C5o | 0.0483 | [0.0335, 0.0632] | [-0.3269, 0.4236] | [-0.3269, 0.4236] (5.0) |
| C8 vs C5o | 0.3975 | [0.3461, 0.4486] | [-0.3894, 1.1844] | [-0.3892, 1.1842] (5.0) |
| C8b vs C5o | 0.1709 | [0.1448, 0.1970] | [-0.2407, 0.5824] | [-0.2406, 0.5824] (5.0) |

### Leave one family out and fold dependence

| comparison | smallest LOFO family p (family omitted) | smallest LOFO SE ratio | fold ICC [upper] | design effect [upper] | SE ratio, fold-adjusted [conservative] |
|---|---|---|---|---|---|
| C4 vs C2 | 0.046 (vignetting) | 5.62 | 0.2293 [0.7142] | 42.05 [128.85] | 2.79 [1.59] |
| C6c vs C2 | 0.045 (vignetting) | 4.83 | 0.0915 [0.4661] | 17.39 [84.44] | 4.88 [2.21] |
| C7 vs C2 | 0.041 (vignetting) | 5.84 | 0.0554 [0.3444] | 10.92 [62.65] | 6.42 [2.68] |
| C7q vs C2 | 0.047 (vignetting) | 6.07 | 0.1200 [0.5386] | 22.48 [97.41] | 3.98 [1.91] |
| C8 vs C2 | 0.208 (shot_noise) | 7.83 | 0.0993 [0.4877] | 18.78 [88.29] | 2.57 [1.19] |
| C8b vs C2 | 0.228 (shot_noise) | 10.21 | 0.0300 [0.2283] | 6.37 [41.87] | 4.61 [1.80] |
| C4 vs C5 | 0.129 (vignetting) | 6.86 | 0.2487 [0.7351] | 45.51 [132.59] | 2.71 [1.59] |
| C6c vs C5 | 0.133 (vignetting) | 6.76 | 0.1965 [0.6734] | 36.17 [121.53] | 3.59 [1.96] |
| C7 vs C5 | 0.126 (vignetting) | 6.77 | 0.1601 [0.6177] | 29.66 [111.57] | 3.08 [1.59] |
| C7q vs C5 | 0.123 (vignetting) | 6.84 | 0.1763 [0.6440] | 32.55 [116.27] | 2.73 [1.44] |
| C8 vs C5 | 0.368 (shot_noise) | 4.15 | 0.1807 [0.6507] | 33.35 [117.48] | 2.72 [1.45] |
| C8b vs C5 | 0.004 (vignetting) | 2.02 | 0.1592 [0.6162] | 29.50 [111.29] | 3.33 [1.71] |
| C4 vs C5o | 0.026 (vignetting) | 5.10 | 0.2922 [0.7754] | 53.30 [139.79] | 2.57 [1.59] |
| C6c vs C5o | 0.021 (vignetting) | 4.31 | 0.1403 [0.5814] | 26.11 [105.07] | 4.21 [2.10] |
| C7 vs C5o | 0.021 (vignetting) | 4.90 | 0.1280 [0.5563] | 23.91 [100.57] | 4.52 [2.20] |
| C7q vs C5o | 0.023 (vignetting) | 5.16 | 0.1928 [0.6683] | 35.51 [120.62] | 3.27 [1.78] |
| C8 vs C5o | 0.214 (shot_noise) | 7.74 | 0.1137 [0.5240] | 21.35 [94.79] | 2.49 [1.18] |
| C8b vs C5o | 0.241 (shot_noise) | 10.03 | 0.0306 [0.2315] | 6.48 [42.43] | 4.75 [1.85] |

Fold component as a share of the family-level variance: C4 vs C2 0.132 [upper 1.112]; C6c vs C2 0.041 [upper 0.351]; C7 vs C2 0.022 [upper 0.201]; C7q vs C2 0.062 [upper 0.532]; C8 vs C2 0.147 [upper 1.267]; C8b vs C2 0.040 [upper 0.384]; C4 vs C5 0.141 [upper 1.181]; C6c vs C5 0.079 [upper 0.667]; C7 vs C5 0.106 [upper 0.895]; C7q vs C5 0.136 [upper 1.147]; C8 vs C5 0.136 [upper 1.152]; C8b vs C5 0.091 [upper 0.769]; C4 vs C5o 0.158 [upper 1.325]; C6c vs C5o 0.056 [upper 0.478]; C7 vs C5o 0.048 [upper 0.414]; C7q vs C5o 0.095 [upper 0.800]; C8 vs C5o 0.159 [upper 1.361]; C8b vs C5o 0.038 [upper 0.362].

Per-comparison tables (leave one family out, per severity, amplitude matched, family x severity means) are in the comparison folders of the output directory.

## NEU-CLS / resnet50.a1_in1k / G1-split1

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split2

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split3

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split4

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split5

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split6

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split7

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split8

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## NEU-CLS / resnet50.a1_in1k / G1-split9

Repeated fold partition, light mode: only the mean, the standard errors and the fold check were computed. See the repeated-partition table at the end.

## Repeated fold partitions

| dataset / backbone / grid | comparison | partitions | SD of the mean over partitions | SE image | SE family | SD / SE image | SD / SE family | SE ratio range |
|---|---|---|---|---|---|---|---|---|
| NEU-CLS / resnet50.a1_in1k / G1 | C4 vs C2 | 10 | 0.0898 | 0.0139 | 0.2185 | 6.45 | 0.41 | 10.91 to 21.72 |
| NEU-CLS / resnet50.a1_in1k / G1 | C6c vs C2 | 10 | 0.1166 | 0.0166 | 0.3077 | 7.05 | 0.38 | 12.61 to 23.18 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7 vs C2 | 10 | 0.0591 | 0.0096 | 0.1727 | 6.13 | 0.34 | 12.78 to 21.99 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7q vs C2 | 10 | 0.0512 | 0.0090 | 0.1399 | 5.71 | 0.37 | 11.14 to 21.59 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8 vs C2 | 10 | 0.1303 | 0.0243 | 0.2557 | 5.36 | 0.51 | 7.61 to 16.91 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8b vs C2 | 10 | 0.0572 | 0.0155 | 0.1653 | 3.69 | 0.35 | 8.16 to 13.44 |
| NEU-CLS / resnet50.a1_in1k / G1 | C4 vs C5 | 10 | 0.0845 | 0.0160 | 0.2511 | 5.30 | 0.34 | 10.59 to 25.34 |
| NEU-CLS / resnet50.a1_in1k / G1 | C6c vs C5 | 10 | 0.1176 | 0.0174 | 0.3358 | 6.77 | 0.35 | 12.14 to 26.93 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7 vs C5 | 10 | 0.0633 | 0.0134 | 0.2041 | 4.72 | 0.31 | 9.79 to 22.32 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7q vs C5 | 10 | 0.0512 | 0.0131 | 0.1765 | 3.90 | 0.29 | 8.91 to 20.16 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8 vs C5 | 10 | 0.1217 | 0.0185 | 0.2483 | 6.56 | 0.49 | 8.42 to 23.25 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8b vs C5 | 10 | 0.0534 | 0.0104 | 0.1530 | 5.13 | 0.35 | 10.03 to 22.43 |
| NEU-CLS / resnet50.a1_in1k / G1 | C4 vs C5o | 10 | 0.0904 | 0.0129 | 0.2074 | 7.01 | 0.44 | 10.15 to 22.95 |
| NEU-CLS / resnet50.a1_in1k / G1 | C6c vs C5o | 10 | 0.1145 | 0.0152 | 0.2955 | 7.55 | 0.39 | 10.44 to 25.39 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7 vs C5o | 10 | 0.0571 | 0.0090 | 0.1601 | 6.37 | 0.36 | 9.03 to 23.77 |
| NEU-CLS / resnet50.a1_in1k / G1 | C7q vs C5o | 10 | 0.0520 | 0.0085 | 0.1290 | 6.15 | 0.40 | 8.84 to 23.11 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8 vs C5o | 10 | 0.1300 | 0.0223 | 0.2375 | 5.84 | 0.55 | 7.08 to 17.78 |
| NEU-CLS / resnet50.a1_in1k / G1 | C8b vs C5o | 10 | 0.0568 | 0.0135 | 0.1462 | 4.22 | 0.39 | 7.19 to 14.40 |


In [20]:
#@title 6.7 Download everything as one zip { display-mode: "form" }
_zip = shutil.make_archive(str(ROOT / "sdic_replication_other900"), "zip", REV)
print(f"archive: {_zip} ({Path(_zip).stat().st_size / 1e6:.1f} MB)")
if DOWNLOAD_ZIP:
    try:
        from google.colab import files as _colab_files
        _colab_files.download(_zip)
    except ImportError:
        print("not running in Colab: copy the archive from the path above")

archive: /content/sdic/sdic_replication_other900.zip (26.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>